In [ ]:
# ============================================================
# LOAD MODULES AND MODEL OUTPUTS
# ============================================================
from Imports import *
from settings import *
from Inputs import *
from Helper import *
from PlotHelper import *
from Outputs import *

# --- Load MODFLOW 6 simulation ---
sim = flopy.mf6.MFSimulation.load(sim_ws=sim_ws, verbosity_level=0)
gwf = sim.get_model(nameModel)

# --- Model arrays from DIS package ---
top     = np.array(gwf.dis.top.array,     dtype=float)
botm    = np.array(gwf.dis.botm.array,    dtype=float)
idomain = np.array(gwf.dis.idomain.array, dtype=int)

# --- Grid properties ---
xorigin = float(gwf.modelgrid.xoffset)
yorigin = float(gwf.modelgrid.yoffset)
delr    = gwf.modelgrid.delr
delc    = gwf.modelgrid.delc
nlay, nrow, ncol = idomain.shape

# --- Head output: memory-efficient load (float32 disk-backed memmap) ---
# Full float64: ~38 GB for 312 Ã— 8 Ã— 1319 Ã— 1527 â†’ MemoryError on most machines
# float32 memmap: ~19 GB on disk; virtually zero RAM; downstream code unchanged.
hds   = bf.HeadFile(os.path.join(sim_ws, f"{nameModel}.hds"))
times = hds.get_times()

def _load_head_cached(sim_ws, nameModel, hds_obj=None, times=None):
    """Load all head timesteps as float32 memmap (disk-backed, low RAM).

    Full float64 load: ~38 GB for 312 periods Ã— 8 layers Ã— 1319 Ã— 1527.
    float32 memmap  : ~19 GB on disk, virtually zero RAM until accessed.
    HDRY (>=1e20) is replaced with NaN during the one-time write pass.
    """
    import numpy as np, flopy.utils.binaryfile as _bf
    _hds = hds_obj or _bf.HeadFile(
        __import__('os').path.join(sim_ws, f"{nameModel}.hds"))
    _times = times if times is not None else _hds.get_times()
    _h0    = _hds.get_data(totim=_times[0])
    _nper  = len(_times)
    _shape = (_nper,) + _h0.shape          # (312, 8, 1319, 1527)
    _cache = __import__('os').path.join(sim_ws, "head_f32_cache.dat")

    if __import__('os').path.exists(_cache):
        _sz_gb = __import__('os').path.getsize(_cache) / 1e9
        print(f"Loading head from disk cache ({_sz_gb:.1f} GB float32 memmap)â€¦")
        return np.memmap(_cache, dtype=np.float32, mode='r', shape=_shape), _times

    _needed_gb = _nper * _h0.size * 4 / 1e9
    print(f"Building float32 head cache  â†’  {_cache}  ({_needed_gb:.1f} GB)")
    _mm = np.memmap(_cache, dtype=np.float32, mode='w+', shape=_shape)
    for _i, _t in enumerate(_times):
        _h = _hds.get_data(totim=_t).astype(np.float32)
        _h[_h >= 1e20] = np.nan
        _mm[_i] = _h
        if (_i + 1) % 52 == 0:
            print(f"  cached {_i+1}/{_nper} periodsâ€¦")
    _mm.flush()
    print("Head cache complete.")
    return _mm, _times

head, times = _load_head_cached(sim_ws, nameModel, hds_obj=hds, times=times)
dates  = pd.date_range(start=START_DATE, periods=len(times), freq="MS")
months = dates

print(f"Model loaded  : {nameModel}")
print(f"Head shape    : {head.shape}")
print(f"Date range    : {dates[0].strftime('%Y-%m')} to {dates[-1].strftime('%Y-%m')}")

# --- Load recharge stress period data ---
# Check permanent rch_cache/ folder first (safe from safe_rmtree),
# then fall back to sim_ws for backward compatibility.
_rch_cache_dir = os.path.join(MODEL_BASE_DIR, "rch_cache")
_rch_perm  = os.path.join(
    _rch_cache_dir,
    f"rch_spd_RCH{RCH_MULT:.2f}_{START_DATE[:7]}_{END_DATE[:7]}.npz"
)
_rch_legacy = os.path.join(sim_ws, "rch_spd.npz")
_rch_path   = _rch_perm if os.path.exists(_rch_perm) else _rch_legacy
if os.path.exists(_rch_path):
    _rch = np.load(_rch_path)
    rch_spd = {int(k): _rch[k] for k in _rch.files}
    print(f"Loaded rch_spd: {len(rch_spd)} periods from:")
    print(f"  {_rch_path}")
else:
    rch_spd = None
    print("WARNING: rch_spd not found -- recharge plots will be skipped")
    print(f"  Checked: {_rch_perm}")
    print(f"  Checked: {_rch_legacy}")


# This notebook will check the Simulations outputs and process the outputs

## 17) Optional: listing-file budget summary

This reads the MF6 listing file and prints the flux budget and percent discrepancy.

In [ ]:
# ---------------------------------------------------------
# MODFLOW 6 budget from listing file  (fast tail approach)
# ---------------------------------------------------------
# The listing file for a 312-period run is ~64 GB.
# flopy.Mf6ListBudget.get_dataframes() parses the ENTIRE file => takes hours.
# This version seeks to the last ~500 KB and prints the final budget table.
# ---------------------------------------------------------
lst_path = os.path.join(sim_ws, f"{nameModel}.lst")
print(f'Listing file: {os.path.getsize(lst_path)/1e9:.2f} GB')

def _read_lst_tail(path, n_bytes=500_000):
    sz = os.path.getsize(path)
    with open(path, 'rb') as _f:
        _f.seek(max(0, sz - n_bytes))
        return _f.read().decode('utf-8', errors='replace')

_tail       = _read_lst_tail(lst_path, n_bytes=500_000)
_tail_lines = _tail.splitlines()

# -- Simulation status --------------------------------------------------------
_term = [l for l in _tail_lines
         if 'termination' in l.lower() or 'Normal' in l or 'ERROR' in l.upper()]
print('=== SIMULATION STATUS ===')
for l in _term:
    print(l)

# -- Last budget table: search back for 'VOLUMETRIC BUDGET' or 'TOTAL IN' -----
_bstart = None
for _i in range(len(_tail_lines) - 1, -1, -1):
    if 'VOLUMETRIC BUDGET' in _tail_lines[_i].upper():
        _bstart = _i
        break
# Fallback: find last 'TOTAL IN' and back up 10 lines for context
if _bstart is None:
    for _i in range(len(_tail_lines) - 1, -1, -1):
        if 'TOTAL IN' in _tail_lines[_i].upper():
            _bstart = max(0, _i - 10)
            break

print('\n=== FINAL PERIOD â€” VOLUMETRIC BUDGET ===')
if _bstart is not None:
    print('\n'.join(_tail_lines[_bstart:_bstart + 70]))
else:
    print('Budget table not found â€” printing last 50 lines of listing file:')
    print('\n'.join(_tail_lines[-50:]))

# -- Percent discrepancy ------------------------------------------------------
_disc = [l.strip() for l in _tail_lines
         if 'PERCENT' in l.upper() and 'DISCREPANCY' in l.upper()]
print('\n=== PERCENT DISCREPANCY ===')
if _disc:
    for l in _disc:
        print(l)
else:
    print('No PERCENT DISCREPANCY lines found â€” increase n_bytes in _read_lst_tail')

# -- Time summary (last period) -----------------------------------------------
_tstart = None
for _i in range(len(_tail_lines) - 1, -1, -1):
    if 'TIME SUMMARY' in _tail_lines[_i].upper():
        _tstart = _i
        break
if _tstart is not None:
    print('\n=== TIME SUMMARY ===')
    print('\n'.join(_tail_lines[_tstart:_tstart + 12]))

# -- Optional full parse (WARNING: very slow on 64 GB file) -------------------
# mf_list = flopy.utils.Mf6ListBudget(lst_path)
# df_flux, df_vol = mf_list.get_dataframes(start_datetime=START_DATE, diff=True)
# print(df_flux.to_string())

## 18) Optional: example post-processing

The original notebook had many exploratory post-processing cells.  
This notebook keeps one representative post-processing figure cell as a starting point.

In [ ]:
# rch_spd path is now resolved automatically in cell 0
# (permanent rch_cache/ folder keyed by RCH_MULT and dates)
# This cell kept for reference -- no action needed.
# Old path (Testing_2): r"D:\Users\abolmaal\modelling\Modflow\Testing_2\rch_spd.npz"


In [ ]:
# ============================================================
# SETTINGS
# ============================================================
save_fig = True
out_fig = os.path.join(fig_dir, "avg_dtw_avg_recharge_studyperiod.png")

# if head is not already loaded â€” use disk-backed float32 memmap (low RAM)
if "head" not in globals() or head is None:
    if "_load_head_cached" not in dir():
        def _load_head_cached(sim_ws, nameModel, hds_obj=None, times=None):
            """Load all head timesteps as float32 memmap (disk-backed, low RAM).
        
            Full float64 load: ~38 GB for 312 periods Ã— 8 layers Ã— 1319 Ã— 1527.
            float32 memmap  : ~19 GB on disk, virtually zero RAM until accessed.
            HDRY (>=1e20) is replaced with NaN during the one-time write pass.
            """
            import numpy as np, flopy.utils.binaryfile as _bf
            _hds = hds_obj or _bf.HeadFile(
                __import__('os').path.join(sim_ws, f"{nameModel}.hds"))
            _times = times if times is not None else _hds.get_times()
            _h0    = _hds.get_data(totim=_times[0])
            _nper  = len(_times)
            _shape = (_nper,) + _h0.shape          # (312, 8, 1319, 1527)
            _cache = __import__('os').path.join(sim_ws, "head_f32_cache.dat")
        
            if __import__('os').path.exists(_cache):
                _sz_gb = __import__('os').path.getsize(_cache) / 1e9
                print(f"Loading head from disk cache ({_sz_gb:.1f} GB float32 memmap)â€¦")
                return np.memmap(_cache, dtype=np.float32, mode='r', shape=_shape), _times
        
            _needed_gb = _nper * _h0.size * 4 / 1e9
            print(f"Building float32 head cache  â†’  {_cache}  ({_needed_gb:.1f} GB)")
            _mm = np.memmap(_cache, dtype=np.float32, mode='w+', shape=_shape)
            for _i, _t in enumerate(_times):
                _h = _hds.get_data(totim=_t).astype(np.float32)
                _h[_h >= 1e20] = np.nan
                _mm[_i] = _h
                if (_i + 1) % 52 == 0:
                    print(f"  cached {_i+1}/{_nper} periodsâ€¦")
            _mm.flush()
            print("Head cache complete.")
            return _mm, _times
    head, times = _load_head_cached(sim_ws, nameModel)
    print("Loaded head array:", head.shape, head.dtype)

# top from model
top = np.array(gwf.dis.top.array, dtype=float)

# ============================================================
# HELPERS
# ============================================================

# ============================================================
# 1) AVERAGE DEPTH TO WATER
# ============================================================
# =========================================================
# COMPUTE DTW â€” with dry cell protection
# =========================================================

# Compute avg_dtw with an incremental running sum â€” avoids allocating a
# (312, 1319, 1527) float64 contiguous block (~4.68 GB) plus its nanmean copy.
# dtw_all is stored as a list of float32 slices (~2.34 GB total, non-contiguous).
_dtw_sum = np.zeros((nrow, ncol), dtype=np.float64)
_dtw_cnt = np.zeros((nrow, ncol), dtype=np.int32)
dtw_all  = []

for i in range(head.shape[0]):
    wt  = get_water_table(head[i], idomain, huge=1e29)
    dtw = np.array(top, dtype=float) - wt

    # hard clip: anything beyond physical bounds is a dry cell artifact
    # Great Lakes Basin: no cell should have DTW > total aquifer thickness (~500 m max)
    dtw = np.where(dtw > 200,  np.nan, dtw)   # dry/inactive artifact
    dtw = np.where(dtw < -50,  np.nan, dtw)   # head >50 m above surface: numerical artifact
    dtw = np.where(idomain[0] <= 0, np.nan, dtw)

    dtw_all.append(dtw.astype(np.float32))   # float32 halves per-slice RAM

    # accumulate for mean â€” no large intermediate array needed
    _valid = np.isfinite(dtw)
    _dtw_sum[_valid] += dtw[_valid]
    _dtw_cnt[_valid] += 1

# average â€” same result as nanmean, no extra 4.68 GB allocation
avg_dtw = np.where(_dtw_cnt > 0, _dtw_sum / _dtw_cnt, np.nan)
del _dtw_sum, _dtw_cnt

# mask inactive cells
avg_dtw = np.where(idomain[0] > 0, avg_dtw, np.nan)

# mask lake cells
if "lake_mask_2d" in globals():
    avg_dtw = np.where(lake_mask_2d, np.nan, avg_dtw)

# ---- final stats ----
finite = avg_dtw[np.isfinite(avg_dtw)]
print("Average DTW after cleaning:")
print(f"  min  : {np.nanmin(finite):.2f} m")
print(f"  p5   : {np.percentile(finite, 5):.2f} m")
print(f"  p50  : {np.percentile(finite, 50):.2f} m")
print(f"  p95  : {np.percentile(finite, 95):.2f} m")
print(f"  max  : {np.nanmax(finite):.2f} m")
print(f"  negative cells : {int(np.sum(finite < 0))}")
print(f"  cells > 50 m   : {int(np.sum(finite > 50)):,}")
print(f"  cells > 100 m  : {int(np.sum(finite > 100)):,}")
# ============================================================
# 2) AVERAGE RECHARGE
# ============================================================
# Auto-load if rch_spd is a file path string (e.g. set by savez call)
if isinstance(rch_spd, str) and rch_spd.endswith(".npz") and os.path.exists(rch_spd):
    _rch = np.load(rch_spd)
    rch_spd = {int(k): _rch[k] for k in _rch.files}
    print(f"Auto-loaded rch_spd: {len(rch_spd)} periods from {rch_spd}")
if rch_spd is None:
    avg_rch = None
    print("WARNING: rch_spd not available â€” recharge subplot will be blank")
elif isinstance(rch_spd, dict):
    rch_arrays = []
    for per in sorted(rch_spd.keys()):
        arr = np.array(rch_spd[per], dtype=float)
        if arr.ndim == 3:
            arr = arr[0]
        rch_arrays.append(arr)
    rch_all  = np.array(rch_arrays, dtype=float)
    avg_rch  = np.nanmean(rch_all, axis=0)
    avg_rch  = np.where(idomain[0] > 0, avg_rch, np.nan)
else:
    arr = np.array(rch_spd, dtype=float)
    if arr.ndim == 3:
        rch_all = arr
    elif arr.ndim == 2:
        rch_all = arr[np.newaxis, :, :]
    else:
        avg_rch = None
        print(f"WARNING: rch_spd shape {getattr(np.array(rch_spd),'shape','?')} unsupported â€” recharge subplot blank")
        rch_all = None
    if rch_all is not None:
        avg_rch = np.nanmean(rch_all, axis=0)
        avg_rch = np.where(idomain[0] > 0, avg_rch, np.nan)

# ============================================================
# 3) COLOR LIMITS
# ============================================================
dtw_vmin, dtw_vmax = robust_limits(avg_dtw, qlow=2, qhigh=98, symmetric=True)
dtw_vmin = -50.0    # artesian / head above land
dtw_vmax =  50.0    # max realistic DTW for Great Lakes Basin
dtw_norm = TwoSlopeNorm(vmin=dtw_vmin, vcenter=0.0, vmax=dtw_vmax)

if avg_rch is not None:
    rch_vmin, rch_vmax = robust_limits(avg_rch, qlow=2, qhigh=98, symmetric=False)
else:
    rch_vmin, rch_vmax = 0.0, 1.0   # dummy â€” subplot will show "not available"

# ============================================================
# 4) BOUNDARY
# ============================================================
gdf_bdry = gpd.read_file(boundary_shp)
try:
    if getattr(gwf.modelgrid, "crs", None) is not None:
        gdf_bdry = gdf_bdry.to_crs(gwf.modelgrid.crs)
except Exception:
    pass

extent = get_extent(xorigin, yorigin, delr, delc, nrow, ncol)

# ============================================================
# 5) PLOT
# ============================================================
# ============================================================
# 5) PLOT
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

# ---- Average DTW
ax = axes[0]
im1 = ax.imshow(
    np.ma.masked_invalid(avg_dtw),
    origin="upper",
    extent=extent,
    cmap="RdBu",
    norm=dtw_norm,
)
gdf_bdry.boundary.plot(ax=ax, color="black", linewidth=0.8)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.text(
    0.02, 0.98, "a)",
    transform=ax.transAxes,
    ha="left", va="top",
    fontsize=14, fontweight="bold"
)
cbar1 = fig.colorbar(im1, ax=ax, shrink=0.82, extend="both")
cbar1.set_label("Average depth to water (m)")

# ---- Average Recharge
ax = axes[1]
if avg_rch is not None:
    im2 = ax.imshow(
        np.ma.masked_invalid(avg_rch),
        origin="upper",
        extent=extent,
        cmap="Blues",
        vmin=rch_vmin,
        vmax=rch_vmax,
    )
    gdf_bdry.boundary.plot(ax=ax, color="black", linewidth=0.8)
    cbar2 = fig.colorbar(im2, ax=ax, shrink=0.82, extend="max")
    cbar2.set_label("Average recharge (m/day)")
else:
    ax.set_facecolor("#eeeeee")
    ax.text(0.5, 0.5, "Recharge not available",
            transform=ax.transAxes, ha="center", va="center",
            fontsize=12, color="#888888")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.text(0.02, 0.98, "b)", transform=ax.transAxes,
        ha="left", va="top", fontsize=14, fontweight="bold")

plt.show()

if save_fig:
    fig.savefig(out_fig, dpi=300, bbox_inches="tight")
    print("Saved:", out_fig)

# ============================================================
# 6) SUMMARY
# ============================================================
print("Average DTW min/max:", np.nanmin(avg_dtw), np.nanmax(avg_dtw))
if avg_rch is not None:
    print("Average recharge min/max:", np.nanmin(avg_rch), np.nanmax(avg_rch))
else:
    print("Average recharge: not available")

In [ ]:

# =========================================================
# 1) Paths to model outputs
# =========================================================
hds_path = Path(sim_ws) / f"{nameModel}.hds"
cbb_path = Path(sim_ws) / f"{nameModel}.cbb"
lst_path = Path(sim_ws) / f"{nameModel}.lst"

print("HDS exists:", hds_path.exists(), hds_path)
print("CBB exists:", cbb_path.exists(), cbb_path)
print("LST exists:", lst_path.exists(), lst_path)

assert hds_path.exists(), f"Head file not found: {hds_path}"


# =========================================================
# 2) Open head file and inspect saved times
# =========================================================
hdobj = flopy.utils.HeadFile(hds_path)
kstpkper_list = hdobj.get_kstpkper()
times = hdobj.get_times()

print("Number of saved times:", len(times))
print("kstpkper list:", kstpkper_list)

extent = get_extent(xorigin, yorigin, delr, delc, nrow, ncol)
date_labels = get_date_labels(len(kstpkper_list), months=months if "months" in globals() else None)


# =========================================================
# 3) Quick diagnostics: raw vs masked ranges
# =========================================================
print("\n--- HEAD DIAGNOSTICS ---")
for i, kp in enumerate(kstpkper_list):
    h = hdobj.get_data(kstpkper=kp)
    h0_raw = np.array(h[0], dtype=float)
    h0 = mask_model_array(h[0], idomain_layer=idomain[0], huge=1e20)

    print(f"{date_labels[i]}:")
    print("   raw min/max   =", np.nanmin(h0_raw), np.nanmax(h0_raw))
    if np.isfinite(h0).any():
        print("   masked min/max=", np.nanmin(h0), np.nanmax(h0))
        print("   finite count  =", np.isfinite(h0).sum())
    else:
        print("   masked array has no finite values")


# =========================================================
# 4) Summary statistics table for layer 1 heads
# =========================================================
stats = []
for i, kp in enumerate(kstpkper_list):
    h = hdobj.get_data(kstpkper=kp)
    h0 = mask_model_array(h[0], idomain_layer=idomain[0], huge=1e20)

    stats.append({
        "stress_period": i + 1,
        "label": date_labels[i],
        "min_head_L1": np.nanmin(h0),
        "max_head_L1": np.nanmax(h0),
        "mean_head_L1": np.nanmean(h0),
        "std_head_L1": np.nanstd(h0),
    })

df_stats = pd.DataFrame(stats)
print("\n--- LAYER 1 HEAD STATS ---")
print(df_stats)


# =========================================================
# 5) Plot layer 1 heads for selected stress periods
# =========================================================
# choose which stress periods to show
plot_ids = [0, 5, 11]   # first, middle, last for a 12-month run
plot_ids = [i for i in plot_ids if i < len(kstpkper_list)]

head_arrays = []
for idx in plot_ids:
    h = hdobj.get_data(kstpkper=kstpkper_list[idx])
    h0 = mask_model_array(h[0], idomain_layer=idomain[0], huge=1e20)
    head_arrays.append(h0)

vmin, vmax = robust_limits(head_arrays, qlow=2, qhigh=98)
print("\nHead plot vmin/vmax:", vmin, vmax)

fig, axes = plt.subplots(1, len(plot_ids), figsize=(6 * len(plot_ids), 5), squeeze=False)

for ax, idx, arr in zip(axes.flat, plot_ids, head_arrays):
    im = ax.imshow(
        np.ma.masked_invalid(arr),
        origin="upper",
        extent=extent,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(f"Layer 1 head\n{date_labels[idx]}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    plt.colorbar(im, ax=ax, shrink=0.8, label="Head (m)")

plt.tight_layout()
plt.show()


# =========================================================
# 6) Plot water table for selected stress periods
# =========================================================
wt_arrays = []
for idx in plot_ids:
    h = hdobj.get_data(kstpkper=kstpkper_list[idx])
    wt = get_water_table(h, idomain, huge=1e20)
    wt_arrays.append(wt)

vmin, vmax = robust_limits(wt_arrays, qlow=2, qhigh=98)
print("Water table plot vmin/vmax:", vmin, vmax)

fig, axes = plt.subplots(1, len(plot_ids), figsize=(6 * len(plot_ids), 5), squeeze=False)

for ax, idx, arr in zip(axes.flat, plot_ids, wt_arrays):
    im = ax.imshow(
        np.ma.masked_invalid(arr),
        origin="upper",
        extent=extent,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(f"Water table\n{date_labels[idx]}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    plt.colorbar(im, ax=ax, shrink=0.8, label="Water table (m)")

plt.tight_layout()
plt.show()


# =========================================================
# 7) Plot head change from first to last stress period
# =========================================================
h_first = hdobj.get_data(kstpkper=kstpkper_list[0])
h_last  = hdobj.get_data(kstpkper=kstpkper_list[-1])

h_first_L1 = mask_model_array(h_first[0], idomain_layer=idomain[0], huge=1e20)
h_last_L1  = mask_model_array(h_last[0],  idomain_layer=idomain[0], huge=1e20)

dh = h_last_L1 - h_first_L1

finite_dh = dh[np.isfinite(dh)]
if finite_dh.size > 0:
    lim = np.nanpercentile(np.abs(finite_dh), 98)
else:
    lim = 1.0

plt.figure(figsize=(8, 6))
plt.imshow(
    np.ma.masked_invalid(dh),
    origin="upper",
    extent=extent,
    cmap="RdBu_r",
    vmin=-lim,
    vmax=lim,
)
plt.colorbar(shrink=0.8, label="Head change (m)")
plt.title(f"Layer 1 head change\n{date_labels[0]} to {date_labels[-1]}")
plt.xlabel("X")
plt.ylabel("Y")
plt.tight_layout()
plt.show()

print("\nHead change min/max (m):", np.nanmin(dh), np.nanmax(dh))



# =========================================================
# 8) Save final figures
# =========================================================
fig_dir = fig_dir
#fig_dir.mkdir(exist_ok=True)

# final water table
wt_last = get_water_table(h_last, idomain, huge=1e20)

plt.figure(figsize=(8, 6))
plt.imshow(
    np.ma.masked_invalid(wt_last),
    origin="upper",
    extent=extent,
    cmap="viridis",
    vmin=np.nanpercentile(wt_last[np.isfinite(wt_last)], 2),
    vmax=np.nanpercentile(wt_last[np.isfinite(wt_last)], 98),
)
plt.colorbar(shrink=0.8, label="Water table (m)")
plt.title(f"Final water table\n{date_labels[-1]}")
plt.xlabel("X")
plt.ylabel("Y")
plt.tight_layout()
plt.savefig(fig_dir / "final_water_table.png", dpi=300, bbox_inches="tight")
plt.show()

# head change
plt.figure(figsize=(8, 6))
plt.imshow(
    np.ma.masked_invalid(dh),
    origin="upper",
    extent=extent,
    cmap="RdBu_r",
    vmin=-lim,
    vmax=lim,
)
plt.colorbar(shrink=0.8, label="Head change (m)")
plt.title(f"Layer 1 head change\n{date_labels[0]} to {date_labels[-1]}")
plt.xlabel("X")
plt.ylabel("Y")
plt.tight_layout()
plt.savefig(fig_dir / "head_change_layer1.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved figures to:", fig_dir)

In [ ]:

# ------------------------------------------------------------
# READ HEAD OUTPUT AFTER A SUCCESSFUL RUN
# ------------------------------------------------------------
headfile = os.path.join(sim_ws, f"{nameModel}.hds")

if not os.path.exists(headfile):
    raise FileNotFoundError(
        f"Head file not found: {headfile}\n"
        "Make sure Cell 12 finished with Run success: True before plotting."
    )

hds = flopy.utils.HeadFile(headfile)

# all output times
times = hds.get_times()
print("Number of output times:", len(times))
print("First/last output time:", times[0], times[-1])

# read all heads â€” float32 disk-backed memmap avoids 38 GB RAM requirement
if "_load_head_cached" not in dir():
    def _load_head_cached(sim_ws, nameModel, hds_obj=None, times=None):
        """Load all head timesteps as float32 memmap (disk-backed, low RAM).
    
        Full float64 load: ~38 GB for 312 periods Ã— 8 layers Ã— 1319 Ã— 1527.
        float32 memmap  : ~19 GB on disk, virtually zero RAM until accessed.
        HDRY (>=1e20) is replaced with NaN during the one-time write pass.
        """
        import numpy as np, flopy.utils.binaryfile as _bf
        _hds = hds_obj or _bf.HeadFile(
            __import__('os').path.join(sim_ws, f"{nameModel}.hds"))
        _times = times if times is not None else _hds.get_times()
        _h0    = _hds.get_data(totim=_times[0])
        _nper  = len(_times)
        _shape = (_nper,) + _h0.shape          # (312, 8, 1319, 1527)
        _cache = __import__('os').path.join(sim_ws, "head_f32_cache.dat")
    
        if __import__('os').path.exists(_cache):
            _sz_gb = __import__('os').path.getsize(_cache) / 1e9
            print(f"Loading head from disk cache ({_sz_gb:.1f} GB float32 memmap)â€¦")
            return np.memmap(_cache, dtype=np.float32, mode='r', shape=_shape), _times
    
        _needed_gb = _nper * _h0.size * 4 / 1e9
        print(f"Building float32 head cache  â†’  {_cache}  ({_needed_gb:.1f} GB)")
        _mm = np.memmap(_cache, dtype=np.float32, mode='w+', shape=_shape)
        for _i, _t in enumerate(_times):
            _h = _hds.get_data(totim=_t).astype(np.float32)
            _h[_h >= 1e20] = np.nan
            _mm[_i] = _h
            if (_i + 1) % 52 == 0:
                print(f"  cached {_i+1}/{_nper} periodsâ€¦")
        _mm.flush()
        print("Head cache complete.")
        return _mm, _times
head, times = _load_head_cached(sim_ws, nameModel, hds_obj=hds, times=times)

# dates matching TDIS monthly periods
dates = pd.date_range(start=START_DATE, periods=head.shape[0], freq="MS")

print("head shape:", head.shape)
print("dates:", dates[0], "to", dates[-1])

# optional plot index for selected maps
max_maps = min(6, head.shape[0])
plot_idx = np.linspace(0, head.shape[0] - 1, max_maps, dtype=int)

# quick sanity check
print("Head min/max:", np.nanmin(np.where(np.abs(head) < 1e20, head, np.nan)),
      np.nanmax(np.where(np.abs(head) < 1e20, head, np.nan)))

In [ ]:
# ============================================================
# SETTINGS
# ============================================================
import math

out_fig_final  = os.path.join(fig_dir, "depthtowater_final_blue_classes.png")
out_fig_annual = os.path.join(fig_dir, "depthtowater_annual_endofyear.png")
out_fig_ts     = os.path.join(fig_dir, "depthtowatertable_annual.png")

# ============================================================
# DEPTH-TO-GROUNDWATER COLOR CLASSES
# ============================================================
bounds = [-30, 0, 7, 25, 50, 100, 250, 500]   # -30â€“0: artesian (head above surface)

colors = [
    "#c62828",  # Artesian  < 0 m   (head above land surface) â€” red
    "#d9f2ff",  # Very shallow  0â€“7 m
    "#8be4ff",  # Shallow       7â€“25 m
    "#2fc7f0",  # Shallow-med  25â€“50 m
    "#10a8d2",  # Medium       50â€“100 m
    "#087f9f",  # Deep        100â€“250 m
    "#045a75",  # Very deep    >250 m
]

dtw_cmap = ListedColormap(colors)
dtw_norm = BoundaryNorm(bounds, dtw_cmap.N)

# ============================================================
# LOAD / RELOAD HEAD  (float32 memmap)
# ============================================================
for var in ["head", "dates", "dtw_all", "dtw_final", "dtw_sel"]:
    if var in globals():
        del globals()[var]

if "_load_head_cached" not in dir():
    def _load_head_cached(sim_ws, nameModel, hds_obj=None, times=None):
        # Load all head timesteps as float32 disk-backed memmap (low RAM)
        import numpy as np, flopy.utils.binaryfile as _bf
        _hds   = hds_obj or _bf.HeadFile(
            __import__('os').path.join(sim_ws, f"{nameModel}.hds"))
        _times = times if times is not None else _hds.get_times()
        _h0    = _hds.get_data(totim=_times[0])
        _nper  = len(_times)
        _shape = (_nper,) + _h0.shape
        _cache = __import__('os').path.join(sim_ws, "head_f32_cache.dat")
        if __import__('os').path.exists(_cache):
            _sz_gb = __import__('os').path.getsize(_cache) / 1e9
            print(f"Loading head cache ({_sz_gb:.1f} GB float32 memmap)â€¦")
            return np.memmap(_cache, dtype=np.float32, mode='r', shape=_shape), _times
        _needed_gb = _nper * _h0.size * 4 / 1e9
        print(f"Building float32 head cache ({_needed_gb:.1f} GB) â†’ {_cache}")
        _mm = np.memmap(_cache, dtype=np.float32, mode='w+', shape=_shape)
        for _i, _t in enumerate(_times):
            _h = _hds.get_data(totim=_t).astype(np.float32)
            _h[_h >= 1e20] = np.nan
            _mm[_i] = _h
            if (_i + 1) % 52 == 0:
                print(f"  cached {_i+1}/{_nper} periodsâ€¦")
        _mm.flush()
        print("Head cache complete.")
        return _mm, _times

head, times = _load_head_cached(sim_ws, nameModel)
dates = pd.date_range(start=START_DATE, periods=head.shape[0], freq="MS")
top   = np.array(gwf.dis.top.array, dtype=float)
print(f"head: {head.shape}  |  {dates[0].strftime('%Y-%m')} â†’ {dates[-1].strftime('%Y-%m')}")

# ============================================================
# MODEL BOUNDARY
# ============================================================
gdf_bdry = gpd.read_file(boundary_shp)
try:
    if getattr(gwf.modelgrid, "crs", None) is not None:
        gdf_bdry = gdf_bdry.to_crs(gwf.modelgrid.crs)
except Exception:
    pass
extent = get_extent(xorigin, yorigin, delr, delc, nrow, ncol)

# ============================================================
# YEAR-END INDICES  (December of each year 2000 â€“ 2025)
# ============================================================
year_end_idx   = [i for i, d in enumerate(dates) if d.month == 12]
year_end_dates = [dates[i] for i in year_end_idx]
n_years        = len(year_end_idx)
print(f"Year-end snapshots: {n_years}  ({year_end_dates[0].year} â€“ {year_end_dates[-1].year})")

# ============================================================
# 1) FINAL DEPTH-TO-WATER MAP  (December 2025)
# ============================================================
dtw_final      = get_depth_to_water(head[-1], idomain, top, clip_negative=False)
dtw_final_plot = np.clip(dtw_final, bounds[0], bounds[-1])

fig, ax = plt.subplots(figsize=(9, 7))
pmv = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
pcm = pmv.plot_array(dtw_final_plot, cmap=dtw_cmap, norm=dtw_norm)
gdf_bdry.boundary.plot(ax=ax, color="black", linewidth=0.8)
ax.grid(False)
ax.set_title(f"Depth to groundwater â€” {dates[-1].strftime('%B %Y')}")
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
add_north_arrow(ax)
add_scale_bar(ax, delr, delc, length_km=SCALEBAR_KM, loc="lower left")
add_dtw_colorbar(fig, pcm, ax, bounds)
plt.tight_layout()
plt.savefig(out_fig_final, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {out_fig_final}")

# ============================================================
# 2) END-OF-YEAR MAPS  (December of each year, 26 panels)
# ============================================================
# Memory strategy:
#   â€¢ plt.ioff()  â€” disables draw_idle() inside the loop (was causing OOM:
#     gdf_bdry.boundary.plot() triggered a full 26-panel render each iteration)
#   â€¢ imshow instead of PlotMapView â€” lighter (no QuadMesh vertex arrays)
#   â€¢ boundary coords pre-extracted once â€” ax.plot() never triggers draw_idle
#   â€¢ dpi=150 and smaller panels â€” the saved PNG is still ~4000 px wide

# Pre-extract boundary line coordinates ONCE (avoids 26Ã— GeoPandas draw calls)
_bdry_lines = []
for _geom in gdf_bdry.geometry:
    _parts = [_geom] if _geom.geom_type == 'Polygon' else list(_geom.geoms)
    for _p in _parts:
        _xs, _ys = _p.exterior.xy
        _bdry_lines.append((list(_xs), list(_ys)))

ncols = 4
nrows = math.ceil(n_years / ncols)      # 7 rows Ã— 4 cols = 28 slots for 26 years

plt.ioff()   # <-- critical: prevents draw_idle() OOM during loop
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(5.5 * ncols, 4.5 * nrows),   # slightly smaller per panel
    constrained_layout=True,
)
axes = np.atleast_1d(axes).ravel()
last_im = None

for k, (ax, idx) in enumerate(zip(axes, year_end_idx)):
    arr      = get_depth_to_water(head[idx], idomain, top, clip_negative=False)
    arr_plot = np.clip(arr, bounds[0], bounds[-1]).astype(np.float32)
    arr_plot[idomain[0] <= 0] = np.nan

    # imshow: lighter than PlotMapView/plot_array (no QuadMesh)
    last_im = ax.imshow(
        np.ma.masked_invalid(arr_plot),
        origin="upper", extent=extent,
        cmap=dtw_cmap, norm=dtw_norm,
        interpolation="nearest",
    )

    # boundary: ax.plot() never triggers draw_idle
    for _xs, _ys in _bdry_lines:
        ax.plot(_xs, _ys, color="black", linewidth=0.5, transform=ax.transData)

    ax.grid(False)
    ax.set_title(f"Dec {dates[idx].year}", fontsize=11, fontweight="bold")
    ax.set_xlabel("X (m)", fontsize=7)
    ax.set_ylabel("Y (m)", fontsize=7)
    ax.tick_params(labelsize=6)

# hide unused panels
for j in range(n_years, len(axes)):
    axes[j].axis("off")

# shared colorbar
add_dtw_colorbar(fig, last_im, axes[:n_years].tolist(), bounds)

fig.suptitle(
    "Great Lakes Basin â€” Depth to Groundwater at End of Each Year (December)",
    fontsize=13, fontweight="bold", y=1.003,
)
plt.savefig(out_fig_annual, dpi=150, bbox_inches="tight")
plt.close(fig)
plt.ion()   # re-enable interactive mode
print(f"Saved: {out_fig_annual}")

# ============================================================
# 3) ANNUAL TIME-SERIES  (December values, 2000 â€“ 2025)
# ============================================================
# Compute spatial statistics for December of each year only
_nper    = head.shape[0]
mean_dtw = np.full(_nper, np.nan)
p05_dtw  = np.full(_nper, np.nan)
p25_dtw  = np.full(_nper, np.nan)
p75_dtw  = np.full(_nper, np.nan)
p95_dtw  = np.full(_nper, np.nan)
median_dtw = np.full(_nper, np.nan)

print(f"Computing DTW stats for {n_years} year-end periodsâ€¦")
for _i in year_end_idx:
    _dtw_i   = get_depth_to_water(head[_i], idomain, top, clip_negative=False)
    _clipped = np.clip(_dtw_i, bounds[0], bounds[-1]).astype(np.float32)
    _flat    = _clipped[np.isfinite(_clipped)]
    if len(_flat) > 0:
        mean_dtw[_i] = float(np.nanmean(_flat))
        p05_dtw[_i]  = float(np.nanpercentile(_flat,  5))
        p25_dtw[_i]  = float(np.nanpercentile(_flat, 25))
        p75_dtw[_i]  = float(np.nanpercentile(_flat, 75))
        p95_dtw[_i]  = float(np.nanpercentile(_flat, 95))
        median_dtw[_i] = float(np.nanmedian(_flat))

# extract only the December values for plotting
_yr_mean = mean_dtw[year_end_idx]
_yr_p05  = p05_dtw[year_end_idx]
_yr_p25  = p25_dtw[year_end_idx]
_yr_p75  = p75_dtw[year_end_idx]
_yr_p95  = p95_dtw[year_end_idx]
_yr_median = median_dtw[year_end_idx]
_yr_yrs  = [d.year for d in year_end_dates]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(_yr_yrs, _yr_median,
        linewidth=2.2, color="steelblue", marker="o", markersize=5,
        label="Median depth to groundwater (December)")

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Depth to groundwater (m)", fontsize=12)
ax.set_title("Great Lakes Basin â€” Annual End-of-Year Depth to Groundwater (December)", fontsize=13)
ax.set_xticks(_yr_yrs)
ax.set_xticklabels(_yr_yrs, rotation=45, ha="right", fontsize=10)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(out_fig_ts, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {out_fig_ts}")

# ============================================================
# SUMMARY
# ============================================================
print(f"\n=== All figures saved to {fig_dir} ===")
print(f"  {out_fig_final}")
print(f"  {out_fig_annual}")
print(f"  {out_fig_ts}")

active      = idomain[0] > 0
valid_final = dtw_final[active]
print("\nFinal DTW stats (Dec 2025, active cells):")
for pct in [5, 25, 50, 75, 95]:
    print(f"  p{pct:2d}: {np.nanpercentile(valid_final, pct):8.1f} m")
print(f"  Negative DTW cells (artesian): "
      f"{int(np.sum(get_depth_to_water(head[-1], idomain, top, clip_negative=False)[active] < 0)):,}")

In [ ]:
# ============================================================
# 4) RECHARGE + DEPTH-TO-WATER  COMBINED TIME SERIES
#    Both variables plotted at December (year-end) snapshots.
#    Top panel   : median recharge (mm/month, green)
#    Bottom panel: median DTW (m, blue)
#    Y-axis for DTW is inverted so shallow water is at the top.
# ============================================================
out_fig_rch_dtw = os.path.join(fig_dir, "recharge_dtw_timeseries.png")

# -- Load rch_spd if needed ---------------------------------------------------
if "rch_spd" not in globals() or rch_spd is None:
    raise RuntimeError("Run the rch_spd cell (ba24f3f1) first.")

_rch_local = rch_spd
if isinstance(_rch_local, str):
    _z = np.load(_rch_local)
    _rch_local = {int(k): _z[k] for k in _z.files}

# -- Active mask (layer 0) ----------------------------------------------------
_active = idomain[0] > 0           # shape (nrow, ncol)

# -- Compute recharge stats at every year-end period --------------------------
# rch_spd keys are stress-period indices (0-based).
# We use the same year_end_idx as DTW (December periods).
_rch_keys_sorted = sorted(_rch_local.keys())

def _rch_stats_for_period(rch_arr_2d, active_mask):
    """Return (median, p25, p75, p05, p95) of recharge over active cells (mm/month)."""
    _flat = rch_arr_2d[active_mask]
    _flat = _flat[np.isfinite(_flat)]
    if len(_flat) == 0:
        return (np.nan,) * 5
    return (
        float(np.nanmedian(_flat)),
        float(np.nanpercentile(_flat, 25)),
        float(np.nanpercentile(_flat, 75)),
        float(np.nanpercentile(_flat,  5)),
        float(np.nanpercentile(_flat, 95)),
    )

_rch_median_mo = []
_rch_p25_mo  = []
_rch_p75_mo  = []
_rch_p05_mo  = []
_rch_p95_mo  = []

for _i in year_end_idx:
    # nearest stress-period key for this time index
    _key = _rch_keys_sorted[min(_i, len(_rch_keys_sorted) - 1)]
    _arr = np.array(_rch_local[_key], dtype=np.float64)
    # rch_spd arrays may be (nlay, nrow, ncol) or (nrow, ncol)
    if _arr.ndim == 3:
        _arr = _arr[0]
    # convert m/day -> mm/month (x1000 x 30.44)
    _mm = _arr * 1000.0 * 30.44
    _m, _q25, _q75, _q05, _q95 = _rch_stats_for_period(_mm, _active)
    _rch_median_mo.append(_m)
    _rch_p25_mo.append(_q25)
    _rch_p75_mo.append(_q75)
    _rch_p05_mo.append(_q05)
    _rch_p95_mo.append(_q95)

_rch_median_mo = np.array(_rch_median_mo)
_rch_p25_mo  = np.array(_rch_p25_mo)
_rch_p75_mo  = np.array(_rch_p75_mo)
_rch_p05_mo  = np.array(_rch_p05_mo)
_rch_p95_mo  = np.array(_rch_p95_mo)

# -- Plot ---------------------------------------------------------------------
_C_DTW = "#1565c0"    # deep blue  -- DTW
_C_RCH = "#2e7d32"    # deep green -- recharge

fig, (ax_rch, ax_dtw) = plt.subplots(2, 1, figsize=(13, 9), sharex=True,
                                      gridspec_kw={"hspace": 0.08})

# -- panel 1 : recharge -------------------------------------------------------
ax_rch.plot(_yr_yrs, _rch_median_mo,
            linewidth=2.2, color=_C_RCH, marker="s", markersize=5,
            label="Median recharge")
ax_rch.set_ylabel("Recharge (mm/month)", fontsize=12, color=_C_RCH)
ax_rch.tick_params(axis="y", labelcolor=_C_RCH, labelsize=10)
ax_rch.set_title(
    "Great Lakes Basin -- Annual Recharge & Depth to Groundwater",
    fontsize=13, fontweight="bold")
ax_rch.legend(fontsize=10, loc="upper right")
ax_rch.grid(axis="y", alpha=0.25)

# -- annotate specific years of interest on both panels ----------------------
_HIGHLIGHT_YEARS = [2013, 2014, 2019, 2020]
_C_HL = "#b71c1c"   # dark red for highlight lines / labels

for _yr_hl in _HIGHLIGHT_YEARS:
    if _yr_hl not in _yr_yrs:
        continue
    _k = _yr_yrs.index(_yr_hl)

    # dashed vertical line spanning both panels
    for _ax in (ax_rch, ax_dtw):
        _ax.axvline(_yr_hl, color=_C_HL, linewidth=1.2,
                    linestyle="--", alpha=0.65, zorder=1)

    # label on recharge panel (above the line value)
    if np.isfinite(_rch_median_mo[_k]):
        ax_rch.annotate(
            str(_yr_hl),
            xy=(_yr_hl, _rch_median_mo[_k]),
            xytext=(0, 10), textcoords="offset points",
            ha="center", fontsize=9.5, color=_C_HL, fontweight="bold",
        )

    # label on DTW panel (below the line value â€” remember axis is inverted)
    if np.isfinite(_yr_median[_k]):
        ax_dtw.annotate(
            str(_yr_hl),
            xy=(_yr_hl, _yr_median[_k]),
            xytext=(0, -14), textcoords="offset points",
            ha="center", fontsize=9.5, color=_C_HL, fontweight="bold",
        )

# -- panel 2 : depth to water -------------------------------------------------
ax_dtw.plot(_yr_yrs, _yr_median,
            linewidth=2.2, color=_C_DTW, marker="o", markersize=5,
            label="Median depth to groundwater (December)")

# invert so shallow (small value) is at top
_dtw_pad = 2
_dtw_min = float(np.nanmin(_yr_median)) - _dtw_pad
_dtw_max = float(np.nanmax(_yr_median)) + _dtw_pad
ax_dtw.set_ylim(_dtw_max, _dtw_min)   # inverted axis

ax_dtw.set_ylabel("Depth to groundwater (m)",
                   fontsize=12, color=_C_DTW)
ax_dtw.tick_params(axis="y", labelcolor=_C_DTW, labelsize=10)
ax_dtw.set_xlabel("Year", fontsize=12)
ax_dtw.set_xticks(_yr_yrs)
ax_dtw.set_xticklabels(_yr_yrs, rotation=45, ha="right", fontsize=10)
ax_dtw.legend(fontsize=10, loc="lower right")
ax_dtw.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(out_fig_rch_dtw, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {out_fig_rch_dtw}")
print(f"Recharge range (Dec, basin median): {float(np.nanmin(_rch_median_mo)):.1f} - {float(np.nanmax(_rch_median_mo)):.1f} mm/month")
print(f"DTW range      (Dec, basin median): {float(np.nanmin(_yr_median)):.1f} - {float(np.nanmax(_yr_median)):.1f} m")

# Compare the Simulation and observartions 

In [ ]:
# ============================================================
# Compare MODFLOW 6 Simulated Heads with Well Observations
#
# SWL = static water level / depth to water below land surface
# Observed head:   obs_head_m  = land_elev_m - SWL_m
# Simulated head:  temporal mean across all stress periods
# ============================================================


# ============================================================
# 1. SIMULATED HEADS: temporal mean over all stress periods
# ============================================================
# head shape: (nper, nlay, nrow, ncol)
# Mean over all periods is more robust than using a single timestep
# for calibration when observation dates are unknown.
h = np.mean(head, axis=0)   # shape: (nlay, nrow, ncol)

print("Mean head shape:", h.shape)
print("Simulation period:", times[0], "to", times[-1])


# ============================================================
# 2. READ WELL DATA
# ============================================================
target_crs = "EPSG:3174"

print("Available layers:")
print(pyogrio.list_layers(wells_gdb_path))

wells = gpd.read_file(
    wells_gdb_path,
    layer=WELL_LAYER,
    engine="pyogrio"
)

boundary = gpd.read_file(boundary_shp)

wells    = wells.to_crs(target_crs)
boundary = boundary.to_crs(target_crs)

# Candidate region column names â€” first one found in the DB will be used
_REGION_CANDIDATES = ["STATE", "PROVINCE", "STATE_PROV", "STATE_CODE",
                      "PROV_CODE", "JURISDICTION", "COUNTRY",
                      "STATE_NAME", "PROVINCE_NAME"]

needed_cols = [
    "WELL_ID", "LAT", "LON",
    "SWL", "WELL_DEPTH",
    "SCREEN_FRM", "SCREEN_TO",
    "AQ_TYPE", "WELL_TYPE",
    "geometry",
]

# Auto-detect which region column the DB actually has
_all_db_cols = list(gpd.read_file(wells_gdb_path, layer=WELL_LAYER,
                                  rows=1, engine="pyogrio").columns)
print("All DB columns:", _all_db_cols)
_region_col = next((c for c in _REGION_CANDIDATES if c in _all_db_cols), None)
if _region_col:
    needed_cols.append(_region_col)
    print(f"Region column found: {_region_col}")
else:
    print("No region column found â€” will assign from LAT/LON")
existing_cols = [c for c in needed_cols if c in wells.columns]
wells = wells[existing_cols].copy()

# â”€â”€ Assign REGION from DB column or LAT/LON bounding boxes â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def _region_from_latlon(lat, lon):
    """Rough state/province from WGS84 coordinates."""
    # Canadian wells: Ontario is north/east of the lakes
    if lon > -76.0:                          return "Ontario"
    if lon > -80.4 and lat > 43.4:           return "Ontario"
    if lon > -83.5 and lat > 46.0:           return "Ontario"
    if lon > -84.5 and lat > 46.2:           return "Ontario"
    # US states
    if lon > -82.4:                          return "Ohio / PA / NY"
    if lon > -84.8 and lat < 42.3:           return "Ohio / Indiana"
    if lat > 45.5 and lon > -90.5:           return "Michigan (UP)"
    if lon > -86.5:                          return "Michigan"
    if lon > -88.1:                          return "Wisconsin / Indiana"
    if lon > -92.9:                          return "Wisconsin / MN"
    return "Unknown"

if _region_col and _region_col in wells.columns:
    wells["region"] = wells[_region_col].astype(str)
elif "LAT" in wells.columns and "LON" in wells.columns:
    wells["region"] = [
        _region_from_latlon(float(r["LAT"]), float(r["LON"]))
        for _, r in wells.iterrows()
    ]
    print("Region assigned from LAT/LON")
else:
    wells["region"] = "Unknown"

print("Region distribution:")
print(wells["region"].value_counts().to_string())

for col in ["SWL", "WELL_DEPTH", "SCREEN_FRM", "SCREEN_TO"]:
    if col in wells.columns:
        wells[col] = pd.to_numeric(wells[col], errors="coerce")

wells = wells.dropna(subset=["SWL", "geometry"]).copy()
print("Original wells after removing missing SWL:", len(wells))


# ============================================================
# 3. CLIP WELLS TO MODEL BOUNDARY
# ============================================================
boundary_union = boundary.geometry.union_all()
wells = wells[wells.geometry.within(boundary_union)].copy()
print("Wells inside model boundary:", len(wells))


# ============================================================
# 4. CLEAN AND CONVERT SWL / WELL DEPTH (feet -> metres)
# ============================================================
wells["SWL_m"] = wells["SWL"] * FT_TO_M

if "WELL_DEPTH" in wells.columns:
    wells["WELL_DEPTH_m"] = wells["WELL_DEPTH"] * FT_TO_M
if "SCREEN_FRM" in wells.columns:
    wells["SCREEN_FRM_m"] = wells["SCREEN_FRM"] * FT_TO_M
if "SCREEN_TO" in wells.columns:
    wells["SCREEN_TO_m"] = wells["SCREEN_TO"] * FT_TO_M

wells = wells[
    (wells["SWL_m"] >= 0) &
    (wells["SWL_m"] < 150)
].copy()
print("Wells after SWL cleaning:", len(wells))


# ============================================================
# 5. GET MODEL ROW/COL -- with out-of-bounds guard
# ============================================================
wells["x_3174"] = wells.geometry.x
wells["y_3174"] = wells.geometry.y

rows, cols, valid = [], [], []
for x, y in zip(wells["x_3174"], wells["y_3174"]):
    try:
        r, c = gwf.modelgrid.intersect(x, y)
        if 0 <= int(r) < nrow and 0 <= int(c) < ncol:
            rows.append(int(r)); cols.append(int(c)); valid.append(True)
        else:
            rows.append(-1); cols.append(-1); valid.append(False)
    except Exception:
        rows.append(-1); cols.append(-1); valid.append(False)

wells["row"]      = rows
wells["col"]      = cols
wells["_in_grid"] = valid
wells = wells[wells["_in_grid"]].drop(columns="_in_grid").copy()
print("Wells inside model grid:", len(wells))

# Land surface elevation (= model top at that cell)
wells["land_elev_m"] = top[
    wells["row"].astype(int),
    wells["col"].astype(int)
]

# Observed groundwater head = land surface minus static water level
wells["obs_head_m"] = wells["land_elev_m"] - wells["SWL_m"]

wells = wells[
    (wells["obs_head_m"] > 0) &
    (wells["obs_head_m"] < 700)
].copy()
print("Wells after observed-head cleaning:", len(wells))


# ============================================================
# 6. ASSIGN MODEL LAYER USING SCREEN OR WELL DEPTH
# ============================================================
if "SCREEN_FRM_m" in wells.columns and "SCREEN_TO_m" in wells.columns:
    wells["screen_mid_m"] = (wells["SCREEN_FRM_m"] + wells["SCREEN_TO_m"]) / 2
else:
    wells["screen_mid_m"] = np.nan

if "WELL_DEPTH_m" in wells.columns:
    wells["screen_mid_m"] = wells["screen_mid_m"].fillna(wells["WELL_DEPTH_m"] / 2)

wells["screen_mid_m"] = wells["screen_mid_m"].fillna(5.0)

layers = []
for _, r in wells.iterrows():
    i_w = int(r["row"])
    j_w = int(r["col"])
    screen_mid_elev = r["land_elev_m"] - r["screen_mid_m"]
    assigned_layer  = 0
    for k in range(botm.shape[0]):
        layer_top = top[i_w, j_w] if k == 0 else botm[k - 1, i_w, j_w]
        layer_bot = botm[k, i_w, j_w]
        if layer_bot <= screen_mid_elev <= layer_top:
            assigned_layer = k
            break
    layers.append(assigned_layer)
wells["layer"] = layers


# ============================================================
# 7. DROP WELLS IN INACTIVE CELLS (idomain <= 0)
# ============================================================
# MF6 inactive cells have no valid simulated head
wells["_idomain"] = [
    int(idomain[int(r["layer"]), int(r["row"]), int(r["col"])])
    for _, r in wells.iterrows()
]
wells = wells[wells["_idomain"] > 0].drop(columns="_idomain").copy()
print("Wells in active cells:", len(wells))



# ============================================================
# 8. BUILD OBSERVATION TABLE AND SAVE
#    Memory-safe version: save geometry-free table for comparison
# ============================================================
import gc

wells = wells.reset_index(drop=True)

if "WELL_ID" in wells.columns:
    wells["well_id"] = wells["WELL_ID"].astype(str)
else:
    wells["well_id"] = [f"WELL_{i:06d}" for i in range(1, len(wells) + 1)]

keep_cols = [
    "well_id", "LAT", "LON", "x_3174", "y_3174",
    "land_elev_m", "SWL", "SWL_m", "obs_head_m",
    "WELL_DEPTH", "WELL_DEPTH_m",
    "SCREEN_FRM", "SCREEN_TO", "SCREEN_FRM_m", "SCREEN_TO_m",
    "screen_mid_m", "AQ_TYPE", "WELL_TYPE",
    "layer", "row", "col", "region"
]

keep_cols = [c for c in keep_cols if c in wells.columns]

# IMPORTANT:
# Do NOT keep geometry in the comparison table.
# Geometry is heavy and not needed for the statistics.
obs_table = pd.DataFrame(wells[keep_cols])

# Reduce memory use
for c in ["LAT", "LON", "x_3174", "y_3174",
          "land_elev_m", "SWL", "SWL_m", "obs_head_m",
          "WELL_DEPTH", "WELL_DEPTH_m",
          "SCREEN_FRM", "SCREEN_TO", "SCREEN_FRM_m", "SCREEN_TO_m",
          "screen_mid_m"]:
    if c in obs_table.columns:
        obs_table[c] = pd.to_numeric(obs_table[c], errors="coerce").astype("float32")

for c in ["layer", "row", "col"]:
    if c in obs_table.columns:
        obs_table[c] = pd.to_numeric(obs_table[c], errors="coerce").astype("int32")

os.makedirs(obs_out_dir, exist_ok=True)
obs_table.to_csv(out_obs_csv, index=False)
print("Saved observation table:", out_obs_csv)

# Free the large GeoDataFrame from memory if you do not need it anymore
del wells
gc.collect()


# ============================================================
# 9. EXTRACT SIMULATED HEAD
# ============================================================
comparison = obs_table

k_arr = comparison["layer"].to_numpy(dtype=np.int32)
i_arr = comparison["row"].to_numpy(dtype=np.int32)
j_arr = comparison["col"].to_numpy(dtype=np.int32)

comparison["sim_head_m"] = h[k_arr, i_arr, j_arr].astype("float32")

# Mask MF6 dry-cell sentinel
comparison.loc[np.abs(comparison["sim_head_m"]) >= 1e20, "sim_head_m"] = np.nan

# Use mask without .copy()
mask = (
    comparison["obs_head_m"].notna() &
    comparison["sim_head_m"].notna() &
    (comparison["obs_head_m"] > 0) &
    (comparison["obs_head_m"] < 700) &
    (comparison["sim_head_m"] > 0) &
    (comparison["sim_head_m"] < 700)
)

comparison = comparison.loc[mask].reset_index(drop=True)

print("Comparison wells after head cleaning:", len(comparison))


# ============================================================
# 10. HEAD COMPARISON STATISTICS
# ============================================================
comparison["residual_m"] = (
    comparison["sim_head_m"] - comparison["obs_head_m"]
).astype("float32")

comparison["abs_error_m"] = comparison["residual_m"].abs().astype("float32")

rmse = np.sqrt(np.mean(comparison["residual_m"].to_numpy(dtype=np.float64) ** 2))
mae  = np.mean(comparison["abs_error_m"].to_numpy(dtype=np.float64))
bias = np.mean(comparison["residual_m"].to_numpy(dtype=np.float64))

print("Number of comparison wells:", len(comparison))
print(f"Head RMSE: {rmse:.2f} m")
print(f"Head MAE : {mae:.2f} m")
print(f"Head Bias: {bias:.2f} m")


# ============================================================
# 11. DEPTH-TO-WATER COMPARISON
# ============================================================
comparison["obs_dtw_m"] = comparison["SWL_m"].astype("float32")
comparison["sim_dtw_m"] = (
    comparison["land_elev_m"] - comparison["sim_head_m"]
).astype("float32")

# Memory-safe DTW filter: no .copy()
dtw_mask = (
    comparison["obs_dtw_m"].notna() &
    comparison["sim_dtw_m"].notna() &
    (comparison["obs_dtw_m"] >= 0) &
    (comparison["obs_dtw_m"] < 150) &
    (comparison["sim_dtw_m"] >= 0) &
    (comparison["sim_dtw_m"] < 150)
)

comparison = comparison.loc[dtw_mask].reset_index(drop=True)

comparison["dtw_residual_m"] = (
    comparison["sim_dtw_m"] - comparison["obs_dtw_m"]
).astype("float32")

dtw_rmse = np.sqrt(np.mean(comparison["dtw_residual_m"].to_numpy(dtype=np.float64) ** 2))
dtw_mae  = np.mean(np.abs(comparison["dtw_residual_m"].to_numpy(dtype=np.float64)))
dtw_bias = np.mean(comparison["dtw_residual_m"].to_numpy(dtype=np.float64))

print("Comparison wells after DTW cleaning:", len(comparison))
print(f"DTW RMSE: {dtw_rmse:.2f} m")
print(f"DTW MAE : {dtw_mae:.2f} m")
print(f"DTW Bias: {dtw_bias:.2f} m")


# ============================================================
# 12. SAVE COMPARISON TABLE
# ============================================================
os.makedirs(compare_out_dir, exist_ok=True)

# Optional: keep only useful output columns to reduce file size
final_cols = [
    "well_id", "LAT", "LON", "x_3174", "y_3174",
    "layer", "row", "col", "region",
    "land_elev_m",
    "SWL_m", "obs_head_m", "sim_head_m",
    "residual_m", "abs_error_m",
    "obs_dtw_m", "sim_dtw_m", "dtw_residual_m",
    "WELL_DEPTH_m", "screen_mid_m",
    "AQ_TYPE", "WELL_TYPE"
]

final_cols = [c for c in final_cols if c in comparison.columns]

comparison[final_cols].to_csv(out_compare_csv, index=False)
print("Saved comparison table:", out_compare_csv)


# ============================================================
# 13. PLOT OBSERVED VS SIMULATED HEAD
# ============================================================
# Sample only for plotting, not for statistics
if len(comparison) > 100000:
    plot_df = comparison.sample(100000, random_state=42)
else:
    plot_df = comparison

plt.figure(figsize=(6, 6))
plt.scatter(
    plot_df["obs_head_m"],
    plot_df["sim_head_m"],
    s=5,
    alpha=0.25
)

min_val = min(comparison["obs_head_m"].min(), comparison["sim_head_m"].min())
max_val = max(comparison["obs_head_m"].max(), comparison["sim_head_m"].max())

plt.plot([min_val, max_val], [min_val, max_val], "k--", linewidth=1)

plt.xlabel("Observed groundwater head (m)")
plt.ylabel("Simulated groundwater head (m)")
plt.title("Observed vs Simulated Groundwater Head")
plt.grid(True, alpha=0.3)

plt.text(
    0.05, 0.95,
    f"RMSE = {rmse:.2f} m\nMAE = {mae:.2f} m\nBias = {bias:.2f} m\nn = {len(comparison):,}",
    transform=plt.gca().transAxes,
    verticalalignment="top",
    bbox=dict(facecolor="white", edgecolor="black", alpha=0.8)
)

plt.tight_layout()
plt.savefig(out_compare_fig, dpi=300)
plt.show()
print("Saved:", out_compare_fig)

# ============================================================
# 14. PLOT OBSERVED VS SIMULATED DEPTH-TO-WATER
# ============================================================
if len(comparison) > 100000:
    _plot_dtw = comparison.sample(100000, random_state=42)
else:
    _plot_dtw = comparison

plt.figure(figsize=(6, 6))
plt.scatter(
    _plot_dtw["obs_dtw_m"],
    _plot_dtw["sim_dtw_m"],
    s=5, alpha=0.25
)
_min_dtw = 0
_max_dtw = max(comparison["obs_dtw_m"].max(), comparison["sim_dtw_m"].max())
plt.plot([_min_dtw, _max_dtw], [_min_dtw, _max_dtw], "k--", linewidth=1)
plt.xlabel("Observed depth to water (m)")
plt.ylabel("Simulated depth to water (m)")
plt.title("Observed vs Simulated Depth to Water")
plt.grid(True, alpha=0.3)
plt.text(
    0.05, 0.95,
    f"RMSE = {dtw_rmse:.2f} m\nMAE = {dtw_mae:.2f} m\nBias = {dtw_bias:.2f} m\nn = {len(comparison):,}",
    transform=plt.gca().transAxes,
    verticalalignment="top",
    bbox=dict(facecolor="white", edgecolor="black", alpha=0.8)
)
plt.tight_layout()
plt.savefig(fig_dir / "dtw_obs_vs_sim.png", dpi=300)
plt.show()
print("Saved:", fig_dir / "dtw_obs_vs_sim.png")


### Per-Layer Observed vs Simulated Head Comparison
Scatter plots, residual histograms, and a statistics table broken down by model layer and region — makes it easy to see which layer drives the RMSE.

In [ ]:
# Reload nameModel from config so titles always reflect the current run
import importlib as _il, sys as _sys
if "config" in _sys.modules:
    _il.reload(_sys.modules["config"])
from config import nameModel  # overrides any stale kernel value

# ============================================================
# PER-LAYER OBSERVED vs SIMULATED HEAD COMPARISON
#
# Three figures:
#   A) Scatter plot: obs vs sim head  — one panel per model layer
#   B) Residual histogram             — one panel per model layer
#   C) Statistics summary table       — bias, RMSE, MAE, n per layer
#                                       and per region
# ============================================================

# ── Load comparison table ─────────────────────────────────────────────────────
if "comparison" not in globals() or comparison is None or len(comparison) == 0:
    if os.path.exists(out_compare_csv):
        comparison = pd.read_csv(out_compare_csv, low_memory=False)
        print(f"Loaded: {len(comparison):,} wells from {out_compare_csv}")
    else:
        raise FileNotFoundError(
            f"Run cell da3b075f first to generate: {out_compare_csv}"
        )

_cmp = comparison.copy()
_cmp["layer_label"] = "Layer " + (_cmp["layer"].astype(int) + 1).astype(str)

_layers   = sorted(_cmp["layer"].dropna().unique().astype(int))
_n_layers = len(_layers)

# Colour palette — one colour per layer
_layer_colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336",
                 "#00BCD4", "#8BC34A", "#FF5722"]

# ── Figure A: per-layer scatter (obs vs sim head) ─────────────────────────────
_ncols = min(_n_layers, 3)
_nrows = (_n_layers + _ncols - 1) // _ncols   # ceiling division

fig_A, axes_A = plt.subplots(
    _nrows, _ncols,
    figsize=(6 * _ncols, 5.5 * _nrows),
    constrained_layout=True,
)
axes_A = np.array(axes_A).ravel()

_stats_rows = []   # collect per-layer stats for table

for _idx, _k in enumerate(_layers):
    _ax  = axes_A[_idx]
    _sub = _cmp[_cmp["layer"] == _k].dropna(
        subset=["obs_head_m", "sim_head_m"]
    )
    _col = _layer_colors[_idx % len(_layer_colors)]

    # Sub-sample for plotting (keep up to 50 000 points per panel)
    _plot = _sub.sample(min(50_000, len(_sub)), random_state=42)

    _ax.scatter(
        _plot["obs_head_m"], _plot["sim_head_m"],
        s=2, alpha=0.20, color=_col, linewidths=0,
    )

    # 1:1 line
    _lo = min(_sub["obs_head_m"].min(), _sub["sim_head_m"].min())
    _hi = max(_sub["obs_head_m"].max(), _sub["sim_head_m"].max())
    _ax.plot([_lo, _hi], [_lo, _hi], "k--", linewidth=1.2, label="1:1")

    # Stats
    _res  = _sub["sim_head_m"] - _sub["obs_head_m"]
    _rmse = float(np.sqrt((_res ** 2).mean()))
    _mae  = float(_res.abs().mean())
    _bias = float(_res.mean())
    _n    = len(_sub)

    _stats_rows.append({
        "Layer": f"Layer {_k + 1}",
        "n": _n,
        "Bias (m)": round(_bias, 2),
        "MAE (m)":  round(_mae,  2),
        "RMSE (m)": round(_rmse, 2),
        "% sim>obs": round((_res > 0).mean() * 100, 1),
    })

    _ax.text(
        0.04, 0.96,
        f"n = {_n:,}\nBias = {_bias:+.2f} m\nRMSE = {_rmse:.2f} m\nMAE  = {_mae:.2f} m",
        transform=_ax.transAxes, va="top", fontsize=8.5,
        bbox=dict(facecolor="white", edgecolor=_col, alpha=0.85, linewidth=1.2),
    )

    _ax.set_xlabel("Observed head (m ASL)", fontsize=10)
    _ax.set_ylabel("Simulated head (m ASL)", fontsize=10)
    _ax.set_title(f"Layer {_k + 1}", fontsize=12, fontweight="bold", color=_col)
    _ax.set_aspect("equal", adjustable="datalim")
    _ax.grid(alpha=0.25)

# Hide unused panels
for _ax in axes_A[_n_layers:]:
    _ax.axis("off")

fig_A.suptitle(
    f"Observed vs Simulated Groundwater Head — by Model Layer\n"
    f"({nameModel}  |  {len(_cmp):,} wells total)",
    fontsize=13, fontweight="bold",
)
_out_A = os.path.join(fig_dir, "obs_vs_sim_by_layer_scatter.png")
fig_A.savefig(_out_A, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", _out_A)

# ── Figure B: per-layer residual histogram ────────────────────────────────────
fig_B, axes_B = plt.subplots(
    _nrows, _ncols,
    figsize=(6 * _ncols, 4.5 * _nrows),
    constrained_layout=True,
)
axes_B = np.array(axes_B).ravel()

for _idx, _k in enumerate(_layers):
    _ax  = axes_B[_idx]
    _sub = _cmp[_cmp["layer"] == _k].dropna(subset=["residual_m"])
    _col = _layer_colors[_idx % len(_layer_colors)]
    _res = _sub["residual_m"].clip(-80, 80)   # clip extremes for display

    _ax.hist(_res, bins=80, color=_col, alpha=0.75, edgecolor="none")
    _ax.axvline(0,  color="black",   linewidth=1.5, linestyle="--", label="Zero")
    _ax.axvline(_res.mean(), color="red", linewidth=1.5,
                linestyle="-", label=f"Bias {_res.mean():+.1f} m")

    _ax.set_xlabel("Residual: sim − obs (m)", fontsize=10)
    _ax.set_ylabel("Well count", fontsize=10)
    _ax.set_title(
        f"Layer {_k + 1}  |  bias = {_sub['residual_m'].mean():+.2f} m  "
        f"|  RMSE = {np.sqrt((_sub['residual_m']**2).mean()):.2f} m",
        fontsize=10, fontweight="bold", color=_col,
    )
    _ax.legend(fontsize=8)
    _ax.grid(alpha=0.25)

for _ax in axes_B[_n_layers:]:
    _ax.axis("off")

fig_B.suptitle(
    f"Residual Distribution (sim − obs) — by Model Layer\n"
    f"({nameModel}  |  clipped to ±80 m for display)",
    fontsize=13, fontweight="bold",
)
_out_B = os.path.join(fig_dir, "residual_histogram_by_layer.png")
fig_B.savefig(_out_B, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", _out_B)

# ── Figure C: statistics summary table ───────────────────────────────────────
_stats_df = pd.DataFrame(_stats_rows)

# Add region breakdown if available
if "region" in _cmp.columns:
    _reg_rows = []
    for _reg, _g in _cmp.groupby("region"):
        _r = _g["residual_m"].dropna()
        if len(_r) < 10:
            continue
        _reg_rows.append({
            "Region": str(_reg),
            "n": len(_r),
            "Bias (m)": round(_r.mean(), 2),
            "MAE (m)":  round(_r.abs().mean(), 2),
            "RMSE (m)": round(np.sqrt((_r**2).mean()), 2),
            "% sim>obs": round((_r > 0).mean() * 100, 1),
        })
    _reg_df = pd.DataFrame(_reg_rows).sort_values("RMSE (m)")
else:
    _reg_df = None

# Add overall row to layer table
_all_r = _cmp["residual_m"].dropna()
_stats_df = pd.concat([
    _stats_df,
    pd.DataFrame([{
        "Layer": "ALL",
        "n": len(_all_r),
        "Bias (m)": round(_all_r.mean(), 2),
        "MAE (m)":  round(_all_r.abs().mean(), 2),
        "RMSE (m)": round(np.sqrt((_all_r**2).mean()), 2),
        "% sim>obs": round((_all_r > 0).mean() * 100, 1),
    }])
], ignore_index=True)

print("\n=== Head comparison statistics by layer ===")
print(_stats_df.to_string(index=False))

if _reg_df is not None:
    print("\n=== Head comparison statistics by region ===")
    print(_reg_df.to_string(index=False))

# Draw as a figure for saving
_n_tables = 2 if _reg_df is not None else 1
fig_C, axes_C = plt.subplots(
    1, _n_tables,
    figsize=(7 * _n_tables, max(3, len(_stats_df) * 0.45 + 1.2)),
    constrained_layout=True,
)
if _n_tables == 1:
    axes_C = [axes_C]

def _fmt_df(df):
    """Format numeric columns to fixed decimals so numpy float64 rounding
    artefacts (e.g. 0.8700000000047...) do not appear in the table."""
    out = df.copy()
    for col in out.columns:
        if col == "n":
            out[col] = out[col].apply(lambda v: f"{int(v):,}")
        elif col in ("Bias (m)", "MAE (m)", "RMSE (m)"):
            out[col] = out[col].apply(lambda v: f"{float(v):+.2f}" if col == "Bias (m)" else f"{float(v):.2f}")
        elif "sim>obs" in col:
            out[col] = out[col].apply(lambda v: f"{float(v):.1f} %")
    return out

def _draw_table(ax, df, title):
    _df = _fmt_df(df)   # apply clean formatting first
    ax.axis("off")
    tbl = ax.table(
        cellText=_df.values,
        colLabels=_df.columns,
        cellLoc="center",
        loc="center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9.5)
    tbl.auto_set_column_width(list(range(len(_df.columns))))
    tbl.scale(1, 1.5)   # taller rows for readability

    # Header row
    for j in range(len(_df.columns)):
        tbl[0, j].set_facecolor("#37474F")
        tbl[0, j].set_text_props(color="white", fontweight="bold")

    # Colour-code ALL cells in the Bias column using ORIGINAL numeric values
    _bias_col = list(df.columns).index("Bias (m)")
    _rmse_col = list(df.columns).index("RMSE (m)")
    for i in range(1, len(df) + 1):
        bias = float(df.iloc[i - 1]["Bias (m)"])
        rmse = float(df.iloc[i - 1]["RMSE (m)"])
        # Bias colour
        if bias > 3:
            tbl[i, _bias_col].set_facecolor("#FFCDD2")   # red  = sim too high
        elif bias < -3:
            tbl[i, _bias_col].set_facecolor("#BBDEFB")   # blue = sim too low
        else:
            tbl[i, _bias_col].set_facecolor("#C8E6C9")   # green = acceptable
        # RMSE colour
        if rmse > 12:
            tbl[i, _rmse_col].set_facecolor("#FFECB3")   # amber = poor
        elif rmse < 7:
            tbl[i, _rmse_col].set_facecolor("#E8F5E9")   # light green = good
        # Alternate row shading for readability
        for j in range(len(_df.columns)):
            if j not in (_bias_col, _rmse_col):
                tbl[i, j].set_facecolor("#FAFAFA" if i % 2 == 0 else "white")

    ax.set_title(title, fontsize=12, fontweight="bold", pad=14)

_draw_table(axes_C[0], _stats_df, f"Statistics by Layer  ({nameModel})")
if _reg_df is not None:
    _draw_table(axes_C[1], _reg_df, "Statistics by Region")

_out_C = os.path.join(fig_dir, "obs_vs_sim_statistics_table.png")
fig_C.savefig(_out_C, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", _out_C)


## Additional Analysis Plots

Cells Aâ€“F add seasonal, spatial, and temporal analyses beyond the basic DTW maps and scatter plots.

In [ ]:
# ============================================================
# A) SEASONAL CYCLE
#    Basin-average DTW for each calendar month (Jan-Dec),
#    averaged over all 25 years.  Shows whether the model
#    captures the spring recharge peak and summer drawdown.
# ============================================================
import calendar

_month_names = [calendar.month_abbr[m] for m in range(1, 13)]
_mean_by_month = []
_p25_by_month  = []
_p75_by_month  = []

_active2d = idomain[0] > 0

print("Computing monthly DTW statistics...")
for _mo in range(1, 13):
    _idx = [i for i, d in enumerate(dates) if d.month == _mo]
    _vals_mo = []
    for _i in _idx:
        _wt  = get_water_table(head[_i], idomain, huge=1e29)
        _dtw = top - _wt
        _dtw = np.where(_active2d & (_dtw > -50) & (_dtw < 200), _dtw, np.nan)
        _vals_mo.append(float(np.nanmean(_dtw)))
    _mean_by_month.append(float(np.nanmean(_vals_mo)))
    _p25_by_month.append(float(np.nanpercentile(_vals_mo, 25)))
    _p75_by_month.append(float(np.nanpercentile(_vals_mo, 75)))
    print(f"  {_month_names[_mo-1]}: mean DTW = {_mean_by_month[-1]:.2f} m")

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(range(1, 13), _p25_by_month, _p75_by_month,
                alpha=0.35, color="steelblue", label="25-75th pct (inter-annual)")
ax.plot(range(1, 13), _mean_by_month,
        color="steelblue", marker="o", linewidth=2.2, markersize=7,
        label="Mean depth to groundwater")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(_month_names, fontsize=11)
ax.set_xlabel("Month", fontsize=12)
ax.set_ylabel("Depth to groundwater (m)", fontsize=12)
ax.set_title("Great Lakes Basin â€” Mean Seasonal Cycle of Depth to Groundwater (2000â€“2025)",
             fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
_out = os.path.join(fig_dir, "seasonal_cycle_dtw.png")
plt.savefig(_out, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", _out)


In [ ]:
# ============================================================
# B) ARTESIAN FREQUENCY MAP
#    For each active land cell: percentage of 312 monthly
#    stress periods where simulated head > land surface.
#    Red patches = persistently artesian.
# ============================================================
_nper_total = head.shape[0]
_artesian_count = np.zeros((nrow, ncol), dtype=np.float32)

print(f"Computing artesian frequency over {_nper_total} periods...")
for _i in range(_nper_total):
    _wt = get_water_table(head[_i], idomain, huge=1e29)
    _artesian_count += (_wt > top).astype(np.float32)
    if (_i + 1) % 60 == 0:
        print(f"  {_i+1}/{_nper_total} done")

_artesian_pct = (_artesian_count / _nper_total) * 100.0
_artesian_pct = np.where(idomain[0] > 0, _artesian_pct, np.nan)

_n_ever = int(np.nansum(_artesian_pct > 0))
_n_always = int(np.nansum(_artesian_pct > 95))
print(f"Cells ever artesian  (>0 %)  : {_n_ever:,}")
print(f"Cells always artesian (>95%) : {_n_always:,}")
print(f"Basin-mean artesian frequency: {np.nanmean(_artesian_pct):.1f} %")

gdf_bdry = gpd.read_file(boundary_shp)

fig, ax = plt.subplots(figsize=(10, 8))
extent = get_extent(xorigin, yorigin, delr, delc, nrow, ncol)
_cmap_art = plt.cm.YlOrRd
im = ax.imshow(np.ma.masked_invalid(_artesian_pct),
               origin="upper", extent=extent,
               cmap=_cmap_art, vmin=0, vmax=100,
               interpolation="nearest")
gdf_bdry.boundary.plot(ax=ax, color="black", linewidth=0.8)
cbar = fig.colorbar(im, ax=ax, shrink=0.75)
cbar.set_label("% of months head > land surface", fontsize=11)
ax.set_title("Artesian Frequency â€” % of Time Head Exceeds Land Surface (2000â€“2025)",
             fontsize=12)
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
add_north_arrow(ax)
add_scale_bar(ax, delr, delc, length_km=SCALEBAR_KM, loc="lower left")
plt.tight_layout()
_out = os.path.join(fig_dir, "artesian_frequency_map.png")
plt.savefig(_out, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", _out)


In [ ]:
# ============================================================
# C) ANNUAL WATER BUDGET
#    Reads the cell-budget file (.cbb) and extracts annual
#    totals for: Recharge IN, Drain OUT, GHB exchange.
#
#    MF6 CBB stores flux RATES (m3/day).  Each record has a
#    'q' field.  Volume = sum(q) * period_length_days.
# ============================================================
import flopy.utils.binaryfile as _bf

_cbb_path = os.path.join(sim_ws, f"{nameModel}.cbb")
print("Reading cell budget file:", _cbb_path)
assert os.path.exists(_cbb_path), f"CBB not found: {_cbb_path}"

_cbb = _bf.CellBudgetFile(_cbb_path)
_record_names = [r.decode().strip() for r in _cbb.get_unique_record_names()]
print("Available budget records:", _record_names)

# â”€â”€ valid (kstp, kper) pairs 
_all_kstpkper = _cbb.get_kstpkper()
_kper_to_kstp = {kper: kstp for kstp, kper in _all_kstpkper}

# period lengths in days â€” read from already-loaded sim object
# (perioddata_run lives in the simulation notebook; use TDIS package instead)
_tdis_pd = sim.tdis.perioddata.array   # structured array: perlen, nstp, tsmult
_perlens = np.array(_tdis_pd["perlen"], dtype=np.float64)
print(f"Period lengths loaded: {len(_perlens)} periods, total {_perlens.sum():.0f} days ({_perlens.sum()/365.25:.1f} years)")

def _find_rec(names, *prefixes):
    for pfx in prefixes:
        for n in names:
            if n.upper().startswith(pfx.upper()):
                return n
    return None

def _read_volume(cbb, recname, nper, perlens):
    # Sum 'q' field over all cells, multiply by period length -> m3/period
    volumes = np.zeros(nper)
    for p in range(nper):
        kstp = _kper_to_kstp.get(p, 0)
        try:
            dat = cbb.get_data(text=recname, kstpkper=(kstp, p))
            if not dat:
                continue
            arr = dat[0]
            # MF6 records are recarrays with a 'q' field (m3/day)
            if hasattr(arr, 'dtype') and arr.dtype.names and 'q' in arr.dtype.names:
                q_rate = float(arr['q'].sum())          # m3/day
            else:
                q_rate = float(np.asarray(arr).sum())   # fallback
            volumes[p] = q_rate * perlens[p]            # m3/period
        except Exception as e:
            pass
    return volumes

_rec_rch = _find_rec(_record_names, "RCHA", "RCH")
_rec_drn = _find_rec(_record_names, "DRN")
_rec_ghb = _find_rec(_record_names, "GHB")
_rec_sto = _find_rec(_record_names, "STO-SS", "STO")

print(f"Records: RCH={_rec_rch}  DRN={_rec_drn}  GHB={_rec_ghb}  STO={_rec_sto}")
_nper_run = len(_perlens)

_rch_vol = _read_volume(_cbb, _rec_rch, _nper_run, _perlens) if _rec_rch else np.zeros(_nper_run)
_drn_vol = _read_volume(_cbb, _rec_drn, _nper_run, _perlens) if _rec_drn else np.zeros(_nper_run)
_ghb_vol = _read_volume(_cbb, _rec_ghb, _nper_run, _perlens) if _rec_ghb else np.zeros(_nper_run)
_sto_vol = _read_volume(_cbb, _rec_sto, _nper_run, _perlens) if _rec_sto else np.zeros(_nper_run)
_cbb.close()

# â”€â”€ annual sums (m3 â†’ km3) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_years = sorted({d.year for d in dates})

def _annual(arr):
    return np.array([arr[[i for i,d in enumerate(dates) if d.year==yr]].sum()
                     for yr in _years])

_rch_yr =  _annual(_rch_vol) / 1e9   # km3/yr
_drn_yr = -_annual(_drn_vol) / 1e9   # drain OUT (negative in CBB â†’ flip)
_ghb_yr =  _annual(_ghb_vol) / 1e9   # net GHB (+ = lakeâ†’gw, - = gwâ†’lake)
_sto_yr =  _annual(_sto_vol) / 1e9   # storage change

print(f"Mean annual recharge : {_rch_yr.mean():+.3f} km3/yr")
print(f"Mean annual drain out: {_drn_yr.mean():+.3f} km3/yr")
print(f"Mean annual GHB net  : {_ghb_yr.mean():+.3f} km3/yr")
print(f"Mean annual dS/dt    : {_sto_yr.mean():+.3f} km3/yr")

# â”€â”€ plot â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(_years))
w = 0.20
ax.bar(x - 1.5*w, _rch_yr,            w, label="Recharge IN",        color="#2e7d32", alpha=0.85)
ax.bar(x - 0.5*w, _drn_yr,            w, label="Drain OUT",          color="#b71c1c", alpha=0.85)
_ghb_pos = np.where(_ghb_yr >= 0, _ghb_yr, 0)
_ghb_neg = np.where(_ghb_yr <  0, _ghb_yr, 0)
ax.bar(x + 0.5*w, _ghb_pos,           w, label="GHB IN (lakeâ†’gw)",   color="#1565c0", alpha=0.85)
ax.bar(x + 0.5*w, _ghb_neg,           w, label="GHB OUT (gwâ†’lake)",  color="#42a5f5", alpha=0.75)
ax.bar(x + 1.5*w, _sto_yr,            w, label="Storage change (Î”S)", color="#6a1b9a", alpha=0.75)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(_years, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Volume (kmÂ³/year)", fontsize=12)
ax.set_title("Great Lakes Basin â€” Annual Water Budget (2000 to2025)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
_out = os.path.join(fig_dir, "annual_water_budget.png")
plt.savefig(_out, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", _out)


In [ ]:
# ============================================================
# D) INDIVIDUAL WELL HYDROGRAPHS
#    Select 6 wells spread across the basin (west-east quantiles).
#    Simulated head = full 312-month time series at well cell/layer.
#    Observed head  = single static value (horizontal dashed line).
# ============================================================

# Load comparison table if not already in memory
if "comparison" not in globals() or comparison is None or len(comparison) == 0:
    _csv = out_compare_csv
    if os.path.exists(_csv):
        comparison = pd.read_csv(_csv)
        print(f"Loaded comparison table: {len(comparison)} wells")
    else:
        raise FileNotFoundError(f"Run cell da3b075f first to generate: {_csv}")

# Select 6 wells: geographically spread west-east, low residual
_df = comparison.copy()
_df["x_c"] = xorigin + (_df["col"] + 0.5) * float(delr[0])
_df["y_c"] = yorigin + (nrow - _df["row"] - 0.5) * float(delc[0])

# Rank wells by x_c, divide into 6 equal-population bands, pick best per band
_df["_xrank"] = pd.qcut(_df["x_c"], 6, labels=False)
_selected = []
for _band in range(6):
    _sub = _df[_df["_xrank"] == _band]
    if len(_sub) == 0:
        continue
    _best = _sub.loc[_sub["abs_error_m"].idxmin()]
    _selected.append(_best)
_selected = _selected[:6]
print(f"Selected {len(_selected)} representative wells")

# Extract simulated time series and plot
fig, axes = plt.subplots(3, 2, figsize=(14, 11), sharex=True, constrained_layout=True)
axes = axes.ravel()

for _ax, _w in zip(axes, _selected):
    _k = int(_w["layer"])
    _i = int(_w["row"])
    _j = int(_w["col"])
    _sim_ts = head[:, _k, _i, _j].astype(np.float64)
    _sim_ts[_sim_ts >= 1e20] = np.nan

    _obs_head = float(_w["obs_head_m"])
    _land     = float(_w["land_elev_m"])
    _res      = float(_w["residual_m"])
    _wid      = str(_w.get("well_id", f"r{_i}c{_j}"))
    _rgn      = str(_w["region"]) if "region" in _w.index else ""

    _ax.plot(dates, _sim_ts,
             color="steelblue", linewidth=1.5, label="Simulated head")
    _ax.axhline(_obs_head, color="#b71c1c", linewidth=1.8,
                linestyle="--", label=f"Observed ({_obs_head:.1f} m)")
    _ax.axhline(_land, color="black", linewidth=0.8,
                linestyle=":", alpha=0.6, label=f"Land surface ({_land:.1f} m)")
    _ax.set_ylabel("Head (m ASL)", fontsize=9)
    _ax.set_title(
        f"{_rgn}  |  Layer {_k+1}  |  ID: {_wid}" + "\n" +
        f"row={_i+1}, col={_j+1}  |  Residual = {_res:+.1f} m",
        fontsize=8.5
    )
    _ax.legend(fontsize=7.5, loc="best")
    _ax.grid(alpha=0.25)
    _ax.tick_params(axis="x", rotation=30, labelsize=8)

for _ax in axes[len(_selected):]:
    _ax.axis("off")

fig.suptitle(
    "Simulated vs Observed Head - 6 Representative Wells (Great Lakes Basin)",
    fontsize=13, fontweight="bold"
)
_out = os.path.join(fig_dir, "well_hydrographs_6wells.png")
plt.savefig(_out, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", _out)


In [ ]:
# ============================================================
# E) SPATIAL CALIBRATION MAP
#    For each model grid cell that contains at least one well,
#    compute the RMSE between observed and simulated heads.
#    Shows WHERE the model performs well and where it struggles.
# ============================================================

if "comparison" not in globals() or len(comparison) == 0:
    _csv = out_compare_csv
    comparison = pd.read_csv(_csv)

# group by grid cell â†’ RMSE and bias
_grp = comparison.groupby(["row", "col"]).agg(
    rmse=("residual_m", lambda x: float(np.sqrt(np.mean(np.asarray(x, dtype=float)**2)))),
    bias=("residual_m", "mean"),
    n_wells=("residual_m", "count"),
).reset_index()

# cell centre coordinates
_grp["x_c"] = xorigin + (_grp["col"] + 0.5) * delr[0]
_grp["y_c"] = yorigin + (nrow - _grp["row"] - 0.5) * delc[0]

print(f"Grid cells with wells: {len(_grp):,}")
print(f"Median RMSE per cell : {_grp['rmse'].median():.2f} m")
print(f"Median Bias per cell : {_grp['bias'].median():+.2f} m")

gdf_bdry = gpd.read_file(boundary_shp)

fig, axes = plt.subplots(1, 2, figsize=(15, 7), constrained_layout=True)

#  left: RMSE 
ax = axes[0]
gdf_bdry.boundary.plot(ax=ax, color="black", linewidth=0.7)
sc = ax.scatter(
    _grp["x_c"], _grp["y_c"],
    c=_grp["rmse"], cmap="RdYlGn_r",
    s=4, vmin=0, vmax=30, linewidths=0
)
fig.colorbar(sc, ax=ax, label="RMSE (m)", shrink=0.8)
ax.set_title("Head RMSE by Grid Cell", fontsize=12)
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")

# â”€â”€ right: Bia
ax = axes[1]
gdf_bdry.boundary.plot(ax=ax, color="black", linewidth=0.7)
_blim = 20
sc2 = ax.scatter(
    _grp["x_c"], _grp["y_c"],
    c=_grp["bias"], cmap="RdBu",
    s=4, vmin=-_blim, vmax=_blim, linewidths=0
)
fig.colorbar(sc2, ax=ax, label="Bias (m)  [+sim > obs]", shrink=0.8)
ax.set_title("Head Bias by Grid Cell  (+ve = model too high)", fontsize=12)
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")

fig.suptitle(
    "Spatial Calibration Performance â€” Observed vs Simulated Head",
    fontsize=13, fontweight="bold"
)
_out = os.path.join(fig_dir, "spatial_calibration_rmse_bias.png")
plt.savefig(_out, dpi=250, bbox_inches="tight")
plt.show()
print("Saved:", _out)


In [ ]:
# ============================================================
# E2) PER-LAYER SPATIAL BIAS MAPS  +  LAYER 5 ARTESIAN DIAGNOSTIC
#
# Key question: is the Layer 5 negative bias (~-7.76 m) near the
# Great Lakes (GHB-controlled) or scattered everywhere?
#
#  - Bias concentrated near lake shores  => GHB stage issue
#  - Bias spatially random               => well assignment issue
# ============================================================
import importlib as _il, sys as _sys
if "config" in _sys.modules:
    _il.reload(_sys.modules["config"])
from config import nameModel  # ensure fresh run name

if "comparison" not in globals() or len(comparison) == 0:
    comparison = pd.read_csv(out_compare_csv)

# ── Per-layer spatial bias (5 panels) ───────────────────────
_layers = sorted(comparison["layer"].unique())
_n_lay  = len(_layers)
_layer_colors = ["#1565C0", "#2E7D32", "#E65100", "#6A1B9A", "#C62828"]

gdf_bdry = gpd.read_file(boundary_shp)

fig_ly, axes_ly = plt.subplots(1, _n_lay,
                               figsize=(5 * _n_lay, 6),
                               constrained_layout=True)
if _n_lay == 1:
    axes_ly = [axes_ly]

for _ax, _lyr, _col in zip(axes_ly, _layers, _layer_colors):
    _sub = comparison[comparison["layer"] == _lyr].copy()
    _grp_ly = _sub.groupby(["row", "col"]).agg(
        bias=("residual_m", "mean"),
        n=("residual_m", "count"),
    ).reset_index()
    _grp_ly["x_c"] = xorigin + (_grp_ly["col"] + 0.5) * delr[0]
    _grp_ly["y_c"] = yorigin + (nrow - _grp_ly["row"] - 0.5) * delc[0]
    _med = float(_grp_ly["bias"].median())
    _blim = max(15.0, float(abs(_grp_ly["bias"].quantile(0.95))))
    gdf_bdry.boundary.plot(ax=_ax, color="black", linewidth=0.5)
    sc = _ax.scatter(
        _grp_ly["x_c"], _grp_ly["y_c"],
        c=_grp_ly["bias"], cmap="RdBu",
        s=3, vmin=-_blim, vmax=_blim, linewidths=0
    )
    fig_ly.colorbar(sc, ax=_ax, label="Bias (m)", shrink=0.75)
    _ax.set_title(
        f"Layer {int(_lyr)}\n"
        f"n={len(_sub):,}  median={_med:+.1f} m",
        fontsize=9, color=_col, fontweight="bold"
    )
    _ax.set_xlabel("X (m)")
    _ax.set_ylabel("Y (m)")

fig_ly.suptitle(
    f"Spatial Bias per Layer  ({nameModel})\n"
    "Blue = model too low  |  Red = model too high",
    fontsize=12, fontweight="bold"
)
_out_ly = os.path.join(fig_dir, "spatial_bias_by_layer.png")
plt.savefig(_out_ly, dpi=220, bbox_inches="tight")
plt.show()
print("Saved:", _out_ly)

# ── Layer 5 artesian diagnostic ─────────────────────────────
print("\n=== LAYER 5 DIAGNOSTICS ===")
_lay5_id = 5 if 5 in _layers else int(max(_layers))
_L5 = comparison[comparison["layer"] == _lay5_id].copy()

if len(_L5) > 0:
    _bias_L5 = float(_L5["residual_m"].mean())
    _rmse_L5 = float(np.sqrt(np.mean(_L5["residual_m"].astype(float)**2)))
    _pct_neg = float((_L5["residual_m"] < 0).mean() * 100)
    _pct_big_neg = float((_L5["residual_m"] < -10).mean() * 100)
    _pct_big_pos = float((_L5["residual_m"] >  10).mean() * 100)
    print(f"  Wells in Layer {_lay5_id}          : {len(_L5):,}")
    print(f"  Mean bias (sim - obs)       : {_bias_L5:+.2f} m")
    print(f"  RMSE                        : {_rmse_L5:.2f} m")
    print(f"  % model < obs (under)       : {_pct_neg:.1f}%")
    print(f"  % model < obs by > 10 m     : {_pct_big_neg:.1f}%  (large under-prediction)")
    print(f"  % model > obs by > 10 m     : {_pct_big_pos:.1f}%  (large over-prediction)")

    if "obs_head_m" in _L5.columns and "sim_head_m" in _L5.columns:
        fig_L5, axes_L5 = plt.subplots(1, 2, figsize=(12, 4),
                                       constrained_layout=True)
        axes_L5[0].hist(_L5["obs_head_m"].dropna(), bins=80,
                        color="#C62828", alpha=0.7, label="Observed")
        axes_L5[0].hist(_L5["sim_head_m"].dropna(), bins=80,
                        color="#1565C0", alpha=0.7, label="Simulated")
        axes_L5[0].set_xlabel("Head (m ASL)")
        axes_L5[0].set_ylabel("Count")
        axes_L5[0].set_title(f"Layer {_lay5_id} Head Distribution")
        axes_L5[0].legend()

        _res_clip = _L5["residual_m"].dropna().clip(-50, 50)
        axes_L5[1].hist(_res_clip, bins=80, color="#6A1B9A", alpha=0.8)
        axes_L5[1].axvline(0, color="black", lw=1.5, label="zero")
        axes_L5[1].axvline(_bias_L5, color="red", lw=1.5, linestyle="--",
                           label=f"Mean = {_bias_L5:+.1f} m")
        axes_L5[1].set_xlabel("Residual (sim - obs) m  [clipped -50 to 50]")
        axes_L5[1].set_ylabel("Count")
        axes_L5[1].set_title(f"Layer {_lay5_id} Residual Distribution")
        axes_L5[1].legend()

        fig_L5.suptitle(
            f"Layer {_lay5_id} Deep Bedrock  ({nameModel})  |  n={len(_L5):,}",
            fontweight="bold"
        )
        _out_L5 = os.path.join(fig_dir, "layer5_head_diagnostics.png")
        plt.savefig(_out_L5, dpi=220, bbox_inches="tight")
        plt.show()
        print("Saved:", _out_L5)
    else:
        print("  (obs_head_m / sim_head_m columns not found -- run cell 12 first)")
else:
    print(f"  No Layer {_lay5_id} wells found in comparison table.")


In [ ]:
# ============================================================
# F) LONG-TERM DTW TREND MAP (2000â€“2025)
#    For each active cell: linear trend in December DTW over
#    26 years, expressed in mm/year.
#    Positive (warm) = water table dropping (drying trend).
#    Negative (cool) = water table rising (wetting trend).
# ============================================================
from scipy import stats as _stats

# year_end_idx selects December periods (fixed earlier)
_n_dec = len(year_end_idx)
_dec_years = np.array([dates[i].year for i in year_end_idx], dtype=np.float64)

print(f"Computing linear trend over {_n_dec} December snapshots...")

# Pre-load all December DTW maps into a small float32 array
# Shape: (n_years, nrow, ncol)  ~  (26, 1319, 1527) â‰ˆ 200 MB
_dec_dtw = np.full((_n_dec, nrow, ncol), np.nan, dtype=np.float32)
for _k, _idx in enumerate(year_end_idx):
    _wt = get_water_table(head[_idx], idomain, huge=1e29)
    _dtw_k = (top - _wt).astype(np.float32)
    _dtw_k[idomain[0] <= 0] = np.nan
    _dtw_k[_dtw_k > 200]    = np.nan
    _dtw_k[_dtw_k < -50]    = np.nan
    _dec_dtw[_k] = _dtw_k
    if (_k+1) % 10 == 0:
        print(f"  {_k+1}/{_n_dec} loaded")

# Per-cell linear trend via vectorised least squares
# slope units: m/year â†’ convert to mm/year
_valid_rows = np.where(np.isfinite(_dec_dtw).any(axis=0))
_trend_map  = np.full((nrow, ncol), np.nan, dtype=np.float32)

_x  = _dec_years - _dec_years.mean()
_sxx = float((_x**2).sum())
for _ii, _jj in zip(*_valid_rows):
    _y = _dec_dtw[:, _ii, _jj].astype(np.float64)
    _ok = np.isfinite(_y)
    if _ok.sum() < 5:
        continue
    _slope = float(np.dot(_x[_ok], _y[_ok])) / float((_x[_ok]**2).sum())
    _trend_map[_ii, _jj] = _slope * 1000.0   # m/yr â†’ mm/yr

print(f"Trend range: {np.nanmin(_trend_map):.1f} to {np.nanmax(_trend_map):.1f} mm/yr")
print(f"Basin median trend: {np.nanmedian(_trend_map):+.1f} mm/yr")

gdf_bdry = gpd.read_file(boundary_shp)
extent = get_extent(xorigin, yorigin, delr, delc, nrow, ncol)

_tlim = float(np.nanpercentile(np.abs(_trend_map[np.isfinite(_trend_map)]), 95))
_tlim = max(_tlim, 5.0)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(
    np.ma.masked_invalid(_trend_map),
    origin="upper", extent=extent,
    cmap="RdBu_r", vmin=-_tlim, vmax=_tlim,
    interpolation="nearest"
)
gdf_bdry.boundary.plot(ax=ax, color="black", linewidth=0.8)
cbar = fig.colorbar(im, ax=ax, shrink=0.75, extend="both")
cbar.set_label("DTW trend (mm/year)+ve = water table dropping", fontsize=11)
ax.set_title("Long-Term Depth-to-Water Trend (December, 2000â€“2025)\n"
    "Warm = drying  |  Cool = wetting",
    fontsize=12
)
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
add_north_arrow(ax)
add_scale_bar(ax, delr, delc, length_km=SCALEBAR_KM, loc="lower left")
plt.tight_layout()
_out = os.path.join(fig_dir, "dtw_longterm_trend.png")
plt.savefig(_out, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", _out)


### E) Stream Gauge Comparison (USGS NWIS)
Downloads USGS daily streamflow, applies Eckhardt baseflow filter, 
and compares basin-total baseflow against simulated DRN+GHB discharge.

In [ ]:
# ============================================================
# E) USGS STREAM GAUGE DOWNLOAD & BASEFLOW COMPARISON
#
# What this does:
#   1. Finds all active USGS daily-streamflow gauges inside the
#      model boundary (US side of Great Lakes Basin)
#   2. Downloads monthly mean streamflow for 2000-2025
#   3. Applies the Eckhardt digital baseflow filter to separate
#      the groundwater-fed portion of streamflow
#   4. Compares basin-total simulated GW discharge (DRN+GHB flux)
#      against observed baseflow  -- both in m3/s
#
# Why:  Well SWL data is a static snapshot; stream baseflow is a
#       continuous signal that directly integrates GW discharge.
# ============================================================
import dataretrieval.nwis as nwis
import geopandas as gpd
from shapely.geometry import Point

# ── 1. Model boundary bounding box (WGS84) for USGS query ────────────────────
_bdry = gpd.read_file(boundary_shp).to_crs("EPSG:4326")
_minx, _miny, _maxx, _maxy = _bdry.total_bounds
print(f"Model bounding box (WGS84): lon {_minx:.2f} to {_maxx:.2f}, lat {_miny:.2f} to {_maxy:.2f}")

# Discover USGS stream gauges -- query by state (bBox+siteType=ST causes 400 error)
# Great Lakes Basin states: MI, WI, OH, IN, IL, PA, NY, MN
import requests, io as _io

_GLB_STATES = ["mi", "wi", "oh", "in", "il", "pa", "ny", "mn"]
_state_dfs  = []
print("Querying USGS NWIS gauges per state...")
for _state in _GLB_STATES:
    _r = requests.get(
        "https://waterservices.usgs.gov/nwis/site/",
        params={
            "format": "rdb",
            "stateCd": _state,
            "parameterCd": "00060",    # streamflow ft3/s
            "siteType": "ST",
        },
        timeout=60,
    )
    if _r.status_code == 200:
        _rdb = [l for l in _r.text.splitlines() if not l.startswith("#")]
        _df  = pd.read_csv(
            _io.StringIO("\n".join(_rdb)), sep="\t",
            skiprows=[1], dtype=str, low_memory=False,
        ).dropna(how="all")
        _state_dfs.append(_df)
        print(f"  {_state.upper()}: {len(_df)} gauges")
    else:
        print(f"  {_state.upper()}: HTTP {_r.status_code} -- skipped")

if not _state_dfs:
    raise RuntimeError("No gauges downloaded -- check internet connection")

_sites_df = pd.concat(_state_dfs, ignore_index=True)
print(f"Gauges in bounding box : {len(_sites_df)}")

# Clip to model boundary polygon
_sites_gdf = gpd.GeoDataFrame(
    _sites_df,
    geometry=gpd.points_from_xy(
        pd.to_numeric(_sites_df["dec_long_va"], errors="coerce"),
        pd.to_numeric(_sites_df["dec_lat_va"],  errors="coerce"),
    ),
    crs="EPSG:4326",
)
_sites_gdf = _sites_gdf.dropna(subset=["dec_lat_va", "dec_long_va"])
_bdry_union = _bdry.geometry.union_all()
_sites_gdf  = _sites_gdf[_sites_gdf.geometry.within(_bdry_union)].copy()
print(f"Gauges inside boundary : {len(_sites_gdf)}")

# Record lengths not available without siteOutput=Expanded;
# gauges with no 2000-2025 data are dropped during the download step.
_gauge_csv = os.path.join(fig_dir, "usgs_gauges_in_boundary.csv")
_sites_gdf.drop(columns="geometry").to_csv(_gauge_csv, index=False)
print(f"Gauge list saved: {_gauge_csv}")

# Download daily streamflow (2000-2025) using the new waterdata API
import dataretrieval.waterdata as _wd
import warnings

_site_ids = list(_sites_gdf["site_no"].astype(str))
# New API requires 'USGS-' prefix on site IDs
_site_ids_prefixed = [f"USGS-{s}" for s in _site_ids]
print(f"Downloading daily flow for {len(_site_ids_prefixed)} gauges (batches of 50)...")

_all_q = []
for _i in range(0, len(_site_ids_prefixed), 50):
    _batch = _site_ids_prefixed[_i:_i+50]
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")   # suppress any remaining deprecation notices
            _dv, _ = _wd.get_daily(
                monitoring_location_id=_batch,
                parameter_code="00060",    # streamflow
                statistic_id="00003",      # daily mean
                time=["2000-01-01", "2025-12-31"],
            )
        if len(_dv) > 0:
            _all_q.append(_dv[["monitoring_location_id", "time", "value"]].copy())
            print(f"  Batch {_i//50+1}: {len(_dv):,} rows for {len(_batch)} sites")
    except Exception as _e:
        print(f"  Batch {_i//50+1} failed: {_e}")

if not _all_q:
    raise RuntimeError("No streamflow data downloaded -- check internet connection")

# Combine and standardise column names to match rest of cell
_q_df = pd.concat(_all_q, ignore_index=True)
_q_df = _q_df.rename(columns={"monitoring_location_id": "site_no", "value": "_Q_cfs"})
_q_df["site_no"]  = _q_df["site_no"].str.replace("USGS-", "", regex=False)
_q_df["datetime"] = pd.to_datetime(_q_df["time"], errors="coerce", utc=True).dt.tz_localize(None)
_q_df["Q_cms"]    = pd.to_numeric(_q_df["_Q_cfs"], errors="coerce") * 0.028317  # ft3/s -> m3/s
_q_df = _q_df.dropna(subset=["datetime", "Q_cms"])


_q_monthly = (
    _q_df.groupby(["site_no", _q_df["datetime"].dt.to_period("M")])["Q_cms"]
    .mean().reset_index()
)
_q_monthly["datetime"] = _q_monthly["datetime"].dt.to_timestamp()

_q_csv = os.path.join(fig_dir, "usgs_monthly_streamflow.csv")
_q_monthly.to_csv(_q_csv, index=False)
print(f"Monthly streamflow saved: {_q_csv}  ({len(_q_monthly):,} rows)")

# ── 4. Eckhardt digital baseflow filter ──────────────────────────────────────
def eckhardt_baseflow(Q, BFI_max=0.80, alpha=0.98):
    """
    Eckhardt (2005) recursive digital baseflow filter.
    Separates groundwater-fed baseflow from total streamflow.
      Q       -- array of streamflow (m3/s), may contain NaN
      BFI_max -- max baseflow index (0.80 = perennial / porous aquifer)
      alpha   -- recession constant (0.98 typical for monthly data)
    Returns baseflow array same shape as Q.
    """
    Q  = np.array(Q, dtype=float)
    bf = np.full_like(Q, np.nan)
    _v = np.where(np.isfinite(Q))[0]
    if len(_v) == 0:
        return bf
    bf[_v[0]] = Q[_v[0]] * BFI_max
    for i in range(_v[0] + 1, len(Q)):
        if not np.isfinite(Q[i]):
            continue
        prev = bf[i-1] if np.isfinite(bf[i-1]) else 0.0
        bf[i] = min(
            ((1 - BFI_max) * alpha * prev + (1 - alpha) * BFI_max * Q[i])
            / (1 - alpha * BFI_max),
            Q[i]
        )
    return bf

_bf_list = []
for _sid, _grp in _q_monthly.groupby("site_no"):
    _g = _grp.sort_values("datetime").copy()
    _g["BF_cms"] = eckhardt_baseflow(_g["Q_cms"].values)
    _bf_list.append(_g)

_q_bf = pd.concat(_bf_list)

# Basin-total (sum all gauges per month)
_basin_monthly = (
    _q_bf.groupby("datetime")[["Q_cms", "BF_cms"]]
    .sum().reset_index()
)
print(f"Basin-total months: {len(_basin_monthly)}")

# ── 5. Read simulated DRN + GHB rates from CBC ───────────────────────────────
_cbb_path = os.path.join(sim_ws, f"{nameModel}.cbb")
_cbb      = bf.CellBudgetFile(_cbb_path)
_tdis_pd  = sim.tdis.perioddata.array
_perlens  = np.array(_tdis_pd["perlen"], dtype=np.float64)

_kper_map = {}
for (_kstp, _kper) in _cbb.get_kstpkper():
    if _kper not in _kper_map:
        _kper_map[_kper] = _kstp

_avail_records = [
    r.strip().decode() if isinstance(r, bytes) else r.strip()
    for r in _cbb.get_unique_record_names()
]
print("CBC records available:", _avail_records)

def _budget_rate_cms(cbb, term, nper, kper_map, avail):
    """Return monthly-mean m3/s for a CBC budget term (positive = out of aquifer)."""
    # Try exact match, then partial match
    # avail may contain bytes (older flopy) or strings -- normalise both
    _avail_str = [r.decode() if isinstance(r, bytes) else r for r in avail]
    _name = term if term in _avail_str else next((r for r in _avail_str if term in r), None)
    if _name is None:
        print(f"  '{term}' not in CBC records -- skipping")
        return np.full(nper, np.nan)
    rates = np.full(nper, np.nan)
    for p in range(nper):
        kstp = kper_map.get(p, 0)
        try:
            dat = cbb.get_data(text=_name, kstpkper=(kstp, p))
            if dat:
                arr = dat[0]
                if hasattr(arr, "dtype") and arr.dtype.names and "q" in arr.dtype.names:
                    q_rate = float(arr["q"].sum())   # m3/day (neg = out of aquifer)
                else:
                    q_rate = float(np.asarray(arr).sum())
                rates[p] = abs(q_rate) / 86400.0    # -> m3/s
        except Exception:
            pass
    return rates

_nper = len(_perlens)
print("Reading DRN...")
_drn_cms = _budget_rate_cms(_cbb, "DRN", _nper, _kper_map, _avail_records)
print("Reading GHB...")
_ghb_cms = _budget_rate_cms(_cbb, "GHB", _nper, _kper_map, _avail_records)

_sim_gw_cms = np.nansum(
    np.column_stack([
        np.where(np.isfinite(_drn_cms), _drn_cms, 0),
        np.where(np.isfinite(_ghb_cms), _ghb_cms, 0)
    ]), axis=1
).astype(float)
_sim_gw_cms[~(np.isfinite(_drn_cms) | np.isfinite(_ghb_cms))] = np.nan

_sim_df = pd.DataFrame({
    "datetime"   : dates,
    "sim_GW_cms" : _sim_gw_cms,
    "sim_DRN_cms": _drn_cms,
    "sim_GHB_cms": _ghb_cms,
})

# ── 6. PLOT ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True, constrained_layout=True)

# Panel A: total streamflow, baseflow, and simulated GW discharge
ax = axes[0]
_obs = _basin_monthly.set_index("datetime").reindex(
    pd.date_range("2000-01-01", "2025-12-01", freq="MS")
)
if not _obs.empty:
    ax.fill_between(_obs.index, _obs["Q_cms"].fillna(0),
                    alpha=0.20, color="royalblue")
    ax.plot(_obs.index, _obs["Q_cms"],
            color="royalblue", linewidth=0.8, alpha=0.6, label="Total streamflow (USGS gauges)")
    ax.fill_between(_obs.index, _obs["BF_cms"].fillna(0),
                    alpha=0.40, color="steelblue")
    ax.plot(_obs.index, _obs["BF_cms"],
            color="steelblue", linewidth=1.4, alpha=0.9, label="Baseflow (Eckhardt filter)")

ax.plot(_sim_df["datetime"], _sim_df["sim_GW_cms"],
        color="#b71c1c", linewidth=2.0,
        label="Simulated GW discharge (DRN + GHB)", zorder=5)
ax.set_ylabel("Flow (m³/s)", fontsize=11)
ax.set_title(
    "Basin-Total Streamflow vs Simulated Groundwater Discharge to Streams",
    fontsize=12, fontweight="bold"
)
ax.legend(fontsize=9, loc="upper right")
ax.grid(alpha=0.25)

# Panel B: DRN vs GHB breakdown
ax2 = axes[1]
ax2.fill_between(_sim_df["datetime"],
                 np.where(np.isfinite(_sim_df["sim_DRN_cms"]), _sim_df["sim_DRN_cms"], 0),
                 alpha=0.5, color="darkorange", label="Drain (DRN)")
ax2.plot(_sim_df["datetime"], _sim_df["sim_DRN_cms"],
         color="darkorange", linewidth=1.3)
ax2.fill_between(_sim_df["datetime"],
                 np.where(np.isfinite(_sim_df["sim_GHB_cms"]), _sim_df["sim_GHB_cms"], 0),
                 alpha=0.45, color="forestgreen", label="General Head Boundary (GHB)")
ax2.plot(_sim_df["datetime"], _sim_df["sim_GHB_cms"],
         color="forestgreen", linewidth=1.3)
ax2.set_ylabel("GW Discharge (m³/s)", fontsize=11)
ax2.set_xlabel("Date", fontsize=10)
ax2.set_title("Simulated GW Discharge Components (2000-2025)", fontsize=12)
ax2.legend(fontsize=9, loc="upper right")
ax2.grid(alpha=0.25)

_out_stream = os.path.join(fig_dir, "stream_obs_vs_sim_baseflow.png")
plt.savefig(_out_stream, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", _out_stream)

# ── 7. Summary statistics: obs baseflow vs sim GW discharge ──────────────────
if not _obs.empty and _obs["BF_cms"].notna().any():
    _obs_bf  = _obs["BF_cms"].values
    _sim_q   = _sim_df.set_index("datetime").reindex(_obs.index)["sim_GW_cms"].values
    _msk     = np.isfinite(_obs_bf) & np.isfinite(_sim_q)
    if _msk.sum() > 5:
        _rmse_q  = np.sqrt(np.mean((_sim_q[_msk] - _obs_bf[_msk])**2))
        _bias_q  = np.mean(_sim_q[_msk] - _obs_bf[_msk])
        _ratio_q = np.mean(_sim_q[_msk]) / np.mean(_obs_bf[_msk])
        print(f"\nBasin-total baseflow comparison ({_msk.sum()} months):")
        print(f"  Gauges included     : {len(_sites_gdf)}")
        print(f"  Obs mean baseflow   : {np.mean(_obs_bf[_msk]):.1f} m3/s")
        print(f"  Sim mean GW disch   : {np.mean(_sim_q[_msk]):.1f} m3/s")
        print(f"  Sim/Obs ratio       : {_ratio_q:.2f}  (1.0 = perfect)")
        print(f"  RMSE                : {_rmse_q:.1f} m3/s")
        print(f"  Bias                : {_bias_q:+.1f} m3/s")


### F) Individual Stream Gauge Comparisons
Observed USGS streamflow (with Eckhardt baseflow separation) vs 
simulated GW discharge (DRN+GHB) at watershed cells for key GLB rivers. 
Each panel covers one watershed; the bounding boxes in `KEY_GAUGES` 
control which DRN cells are summed for each river.

In [ ]:
# ============================================================
# F) INDIVIDUAL STREAM GAUGE COMPARISONS
#    Key rivers in the Great Lakes Basin
#
#    Observed : USGS daily flow -> monthly mean -> Eckhardt baseflow
#    Simulated: DRN flux summed over watershed cells (m3/s)
#              (GHB excluded -- it represents lake exchange, not streams)
#
#    Note: The model only simulates groundwater discharge to streams
#    (baseflow), NOT surface runoff. Observed baseflow is separated
#    using the Eckhardt recursive digital filter for fair comparison.
# ============================================================
import requests, io as _io, warnings
import dataretrieval.waterdata as _wd
import geopandas as gpd
from pyproj import Transformer

# ── User-configurable gauge list ─────────────────────────────────────────────
# Add or remove rivers here. bbox = (lon_min, lat_min, lon_max, lat_max) WGS84
# covering the upstream watershed above the gauge.
KEY_GAUGES = [
    # Michigan -- Lower Peninsula
    {"name": "Grand River, MI",        "site": "04119000", "bbox": (-86.50, 42.50, -84.80, 43.90)},
    {"name": "Saginaw River, MI",      "site": "04157000", "bbox": (-84.50, 42.80, -83.00, 44.50)},
    {"name": "Muskegon River, MI",     "site": "04122500", "bbox": (-86.10, 43.50, -84.60, 44.60)},
    {"name": "Au Sable River, MI",     "site": "04136900", "bbox": (-84.80, 44.20, -83.30, 44.90)},
    {"name": "Kalamazoo River, MI",    "site": "04108660", "bbox": (-86.30, 42.00, -84.40, 42.70)},
    {"name": "St. Joseph River, MI",   "site": "04101500", "bbox": (-86.50, 41.20, -84.60, 42.10)},
    {"name": "Huron River, MI",        "site": "04174500", "bbox": (-84.30, 41.90, -83.20, 42.80)},
    {"name": "Tittabawassee R., MI",   "site": "04145000", "bbox": (-84.80, 43.50, -83.80, 44.00)},
    {"name": "Black River, MI",        "site": "04141000", "bbox": (-84.50, 44.80, -83.70, 45.40)},
    # Michigan -- Upper Peninsula
    {"name": "Menominee River, WI/MI", "site": "04066003", "bbox": (-89.00, 45.00, -87.60, 46.10)},
    {"name": "Escanaba River, MI",     "site": "04057004", "bbox": (-87.40, 45.60, -86.80, 46.40)},
    # Wisconsin
    {"name": "Fox River, WI",          "site": "04084500", "bbox": (-89.20, 42.90, -87.80, 44.90)},
    {"name": "Milwaukee River, WI",    "site": "04087000", "bbox": (-88.50, 42.90, -87.90, 43.50)},
    {"name": "Sheboygan River, WI",    "site": "04086000", "bbox": (-88.50, 43.50, -87.70, 44.20)},
    # Ohio
    {"name": "Maumee River, OH",       "site": "04193500", "bbox": (-85.50, 40.50, -83.40, 42.20)},
    {"name": "Sandusky River, OH",     "site": "04198000", "bbox": (-83.50, 40.50, -82.50, 41.50)},
    {"name": "Cuyahoga River, OH",     "site": "04208000", "bbox": (-82.10, 41.00, -81.20, 41.60)},
    {"name": "Black River, OH",        "site": "04112500", "bbox": (-82.40, 40.90, -81.80, 41.60)},
    # Indiana
    {"name": "Elkhart River, IN",      "site": "04099000", "bbox": (-86.10, 41.00, -85.30, 41.90)},
    # New York
    {"name": "Genesee River, NY",      "site": "04231600", "bbox": (-78.20, 42.10, -77.40, 43.40)},
    {"name": "Oswego River, NY",       "site": "04240300", "bbox": (-76.70, 42.80, -75.50, 43.90)},
    {"name": "Cattaraugus Ck., NY",    "site": "04213500", "bbox": (-79.20, 42.00, -78.70, 42.60)},
    # Minnesota / Wisconsin border
    {"name": "Brule River, WI",        "site": "04025500", "bbox": (-92.00, 45.80, -91.30, 46.60)},
]
# ── Eckhardt baseflow filter (reuse or redefine) ──────────────────────────────
def _eckhardt(Q, BFI_max=0.80, alpha=0.98):
    Q  = np.array(Q, dtype=float)
    bf = np.full_like(Q, np.nan)
    v  = np.where(np.isfinite(Q))[0]
    if not len(v):
        return bf
    bf[v[0]] = Q[v[0]] * BFI_max
    for k in range(v[0]+1, len(Q)):
        if not np.isfinite(Q[k]):
            continue
        prev = bf[k-1] if np.isfinite(bf[k-1]) else 0.0
        bf[k] = min(
            ((1-BFI_max)*alpha*prev + (1-alpha)*BFI_max*Q[k]) / (1-alpha*BFI_max),
            Q[k]
        )
    return bf

# ── Build transformer: WGS84 -> model CRS (EPSG:3174) ────────────────────────
_tfm = Transformer.from_crs("EPSG:4326", f"EPSG:{EPSG}", always_xy=True)

_gauge_results = {}  # per-gauge results for diagnostic cell

# ── Read DRN cell coordinates from the model ─────────────────────────────────
# DRN stress-period data: recarray with cellid=(layer, row, col), elev, cond
print("Reading DRN cell coordinates from model...")
_drn_spd = gwf.drn.stress_period_data.get_data(key=0)

# cellid field: tuple (k, i, j) in MF6 structured grid
# Convert to x/y model coordinates (cell centres, EPSG:3174)
_drn_k = np.array([c[0] for c in _drn_spd["cellid"]], dtype=np.int32)
_drn_i = np.array([c[1] for c in _drn_spd["cellid"]], dtype=np.int32)
_drn_j = np.array([c[2] for c in _drn_spd["cellid"]], dtype=np.int32)

# MF6 node index (0-based): node = k*nrow*ncol + i*ncol + j
_drn_node = (_drn_k * nrow * ncol + _drn_i * ncol + _drn_j).astype(np.int64)

# Cell-centre coordinates in EPSG:3174
_drn_x = xorigin + (_drn_j + 0.5) * float(delr[0])
_drn_y = yorigin + (nrow - _drn_i - 0.5) * float(delc[0])
print(f"  Total DRN cells: {len(_drn_node):,}")

# Pre-compute shapely point array for fast vectorised containment checks
import shapely as _shp
_drn_geoms = _shp.points(_drn_x, _drn_y)   # shapely 2.x vectorised points

# Download exact upstream watershed boundaries from USGS NLDI API.
# Rectangular bboxes over-count DRN cells from adjacent watersheds;
# NLDI returns the precise upstream drainage polygon for each gauge.
from shapely.geometry import shape as _shp_shape
from shapely.ops import transform as _shp_transform
import pyproj as _pyproj
_proj_wgs_to_model = _pyproj.Transformer.from_crs(
    "EPSG:4326", f"EPSG:{EPSG}", always_xy=True
)

_basin_polys = {}   # site_no -> shapely Polygon in model CRS (or None)
print("\nDownloading watershed polygons from USGS NLDI...")
for _g in KEY_GAUGES:
    _sno = _g["site"]
    _nldi_url = (
        "https://labs.waterdata.usgs.gov/api/nldi/linked-data"
        f"/nwissite/USGS-{_sno}/basin"
    )
    try:
        _r = requests.get(_nldi_url, timeout=30)
        _r.raise_for_status()
        _geom_wgs = _shp_shape(_r.json()["features"][0]["geometry"])
        _geom_mod = _shp_transform(_proj_wgs_to_model.transform, _geom_wgs)
        _basin_polys[_sno] = _geom_mod
        print(f"  {_g['name']}: {_geom_mod.area/1e6:,.0f} km2 upstream")
    except Exception as _nldi_e:
        _basin_polys[_sno] = None
        print(f"  {_g['name']}: NLDI failed ({_nldi_e}) -- using bbox fallback")


# ── Open CBB (reuse if already open, else open it) ────────────────────────────
_cbb_path = os.path.join(sim_ws, f"{nameModel}.cbb")
_cbb_g    = bf.CellBudgetFile(_cbb_path)
_avail_g  = [r.strip().decode() if isinstance(r, bytes) else r.strip()
             for r in _cbb_g.get_unique_record_names()]
_kper_g   = {}
for (_kstp, _kper) in _cbb_g.get_kstpkper():
    if _kper not in _kper_g:
        _kper_g[_kper] = _kstp

_tdis_pd  = sim.tdis.perioddata.array
_perlens  = np.array(_tdis_pd["perlen"], dtype=np.float64)
_nper     = len(_perlens)

def _read_term_nodes(cbb, term, avail, nper, kper_map, node_mask):
    """
    Read a CBC budget term, filter to node_mask, sum flux per period.
    Returns array of m3/s values (rate averaged over the period).
    node_mask: boolean array aligned with the full CBB record order,
               OR a set of node integers to filter by.
    """
    _nm = term if term in avail else next((r for r in avail if term in r), None)
    if _nm is None:
        return np.full(nper, np.nan)
    rates = np.full(nper, np.nan)
    for p in range(nper):
        kstp = kper_map.get(p, 0)
        try:
            dat = cbb.get_data(text=_nm, kstpkper=(kstp, p))
            if not dat:
                continue
            arr = dat[0]
            if not (hasattr(arr, "dtype") and arr.dtype.names):
                continue
            # Filter to watershed nodes
            if "node" in arr.dtype.names:
                # MF6 CBB: 1-based node, DRN nodes are 0-based -> add 1
                _mask = np.isin(arr["node"], node_mask)
            elif "node" in [f.lower() for f in arr.dtype.names]:
                _nf   = [f for f in arr.dtype.names if f.lower()=="node"][0]
                _mask = np.isin(arr[_nf], node_mask)
            else:
                _mask = np.ones(len(arr), dtype=bool)   # fallback: use all

            if "q" in arr.dtype.names:
                q_day = float(arr["q"][_mask].sum())     # m3/day
            else:
                q_day = float(np.asarray(arr)[_mask].sum())
            rates[p] = abs(q_day) / 86400.0              # m3/day -> m3/s
        except Exception:
            pass
    return rates

# ── Download observed flow for all gauges (one batch) ────────────────────────
_all_sites = ["USGS-" + g["site"] for g in KEY_GAUGES]
print(f"Downloading USGS flow for {len(_all_sites)} gauges...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    try:
        _obs_raw, _ = _wd.get_daily(
            monitoring_location_id=_all_sites,
            parameter_code="00060",
            statistic_id="00003",
            time=["2000-01-01", "2025-12-31"],
        )
        _obs_raw = _obs_raw[["monitoring_location_id","time","value"]].copy()
        _obs_raw["site_no"]  = _obs_raw["monitoring_location_id"].str.replace("USGS-","",regex=False)
        _obs_raw["datetime"] = pd.to_datetime(_obs_raw["time"],errors="coerce",utc=True).dt.tz_localize(None)
        _obs_raw["Q_cms"]    = pd.to_numeric(_obs_raw["value"],errors="coerce") * 0.028317
        _obs_raw = _obs_raw.dropna(subset=["datetime","Q_cms"])
        print(f"  Downloaded {len(_obs_raw):,} daily records")
    except Exception as _e:
        print(f"  Download failed: {_e}")
        _obs_raw = pd.DataFrame(columns=["site_no","datetime","Q_cms"])

# ── Plot: one panel per gauge ─────────────────────────────────────────────────
_ncols = 2
_nrows = (len(KEY_GAUGES) + 1) // 2
fig, axes = plt.subplots(_nrows, _ncols,
                         figsize=(14, 4.5 * _nrows),
                         sharex=True, constrained_layout=True)
axes = np.array(axes).ravel()

_colors_obs = "#1565C0"   # dark blue  = observed baseflow
_colors_sim = "#b71c1c"   # dark red   = simulated GW discharge

for _gi, _gauge in enumerate(KEY_GAUGES):
    _ax   = axes[_gi]
    _name = _gauge["name"]
    _site = _gauge["site"]
    _bbox = _gauge["bbox"]   # (lon_min, lat_min, lon_max, lat_max)

    # ── Find DRN cells inside the watershed bounding box ──────────────────
    # Use actual upstream watershed polygon (NLDI) to filter DRN cells.
    # Falls back to rectangular bbox only if NLDI download failed.
    _basin_poly = _basin_polys.get(_site)
    if _basin_poly is not None:
        # shapely 2.x vectorised containment -- fast C-level check
        _in_ws = _shp.within(_drn_geoms, _basin_poly)
        _method = "NLDI polygon"
    else:
        # Fallback to rectangular bbox
        _x0, _y0 = _tfm.transform(_bbox[0], _bbox[1])
        _x1, _y1 = _tfm.transform(_bbox[2], _bbox[3])
        _xmin, _xmax = min(_x0, _x1), max(_x0, _x1)
        _ymin, _ymax = min(_y0, _y1), max(_y0, _y1)
        _in_ws = ((_drn_x >= _xmin) & (_drn_x <= _xmax) &
                  (_drn_y >= _ymin) & (_drn_y <= _ymax))
        _method = "bbox fallback"
    _ws_nodes_0based = _drn_node[_in_ws]
    # CBB uses 1-based nodes in MF6
    _ws_nodes_1based = _ws_nodes_0based + 1
    print(f"\n{_name}: {_in_ws.sum():,} DRN cells ({_method})")

    # ── Observed flow: monthly mean -> Eckhardt baseflow ──────────────────
    _site_df = _obs_raw[_obs_raw["site_no"] == _site].copy()
    if len(_site_df) == 0:
        print(f"  No observed data for {_site}")
        _obs_monthly = pd.DataFrame()
    else:
        _mth = (_site_df.groupby(_site_df["datetime"].dt.to_period("M"))["Q_cms"]
                .mean().reset_index())
        _mth["datetime"] = _mth["datetime"].dt.to_timestamp()
        _mth = _mth.sort_values("datetime")
        _mth["BF_cms"] = _eckhardt(_mth["Q_cms"].values)
        _obs_monthly = _mth

    # ── Simulated: DRN + GHB flux at watershed cells ──────────────────────
    if len(_ws_nodes_1based) == 0:
        _sim_cms = np.full(_nper, np.nan)
        print(f"  WARNING: no DRN cells found for {_name} — check bbox")
    else:
        # DRN-only: GHB represents exchange with the Great Lakes,
        # NOT drainage to rivers.  Including GHB inflates the
        # simulated streamflow by adding lake-shoreline flux.
        _sim_cms = _read_term_nodes(_cbb_g, "DRN", _avail_g,
                                    _nper, _kper_g, _ws_nodes_1based)

    # ── Plot ───────────────────────────────────────────────────────────────
    if len(_obs_monthly) > 0:
        _ax.fill_between(_obs_monthly["datetime"], _obs_monthly["Q_cms"],
                         alpha=0.12, color=_colors_obs)
        _ax.plot(_obs_monthly["datetime"], _obs_monthly["Q_cms"],
                 color=_colors_obs, linewidth=0.8, alpha=0.5,
                 label="Total streamflow (USGS)")
        _ax.plot(_obs_monthly["datetime"], _obs_monthly["BF_cms"],
                 color=_colors_obs, linewidth=1.6,
                 label="Baseflow (Eckhardt filter)")

    _ax.plot(dates, _sim_cms,
             color=_colors_sim, linewidth=1.8,
             label="Simulated GW discharge (DRN only)")

    # Stats (over overlapping period)
    if len(_obs_monthly) > 0 and not np.all(np.isnan(_sim_cms)):
        _sim_df_m = pd.DataFrame({"datetime": dates, "sim": _sim_cms})
        _merged   = _obs_monthly.set_index("datetime")["BF_cms"].reindex(
            _sim_df_m.set_index("datetime").index
        )
        _msk = np.isfinite(_sim_cms) & np.isfinite(_merged.values)
        if _msk.sum() > 6:
            _ratio = np.mean(_sim_cms[_msk]) / np.mean(_merged.values[_msk])
            _rmse  = np.sqrt(np.mean((_sim_cms[_msk]-_merged.values[_msk])**2))
            _ax.text(0.02, 0.97,
                     f"Sim/Obs baseflow = {_ratio:.2f}\nRMSE = {_rmse:.1f} m³/s",
                     transform=_ax.transAxes, va="top", fontsize=8.5,
                     bbox=dict(facecolor="white", edgecolor="gray",
                               alpha=0.85, linewidth=0.8))

    _ax.set_ylabel("Streamflow (m³/s)", fontsize=9)
    _ax.set_title(_name, fontsize=11, fontweight="bold")
    _ax.legend(fontsize=8, loc="upper right")
    _ax.grid(alpha=0.25)
    _ax.tick_params(axis="x", rotation=25, labelsize=8)

    # Save for diagnostic cell
    _gauge_results[_site] = {
        "name": _name,
        "obs":  _obs_monthly.copy() if len(_obs_monthly) > 0 else None,
        "sim":  _sim_cms.copy(),
        "n_drn": int(_in_ws.sum()),
    }

# Hide unused panels
for _ax in axes[len(KEY_GAUGES):]:
    _ax.axis("off")

fig.suptitle(
    f"Observed Streamflow vs Simulated GW Discharge (DRN only) — Key GLB Rivers\n"
    f"({nameModel}  |  Observed baseflow separated with Eckhardt filter)",
    fontsize=13, fontweight="bold"
)
_out_gauges = os.path.join(fig_dir, "stream_gauges_individual.png")
plt.savefig(_out_gauges, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", _out_gauges)


### G) Stream Gauge Diagnostic: NSE, KGE, Seasonal RMSE, Lag
Runs after Cell F. Uses `_gauge_results` to compute scale-free metrics so
you can see WHY RMSE is high: timing offset, magnitude bias, or variability mismatch.

In [ ]:
# ============================================================
# G) STREAM GAUGE DIAGNOSTIC SUMMARY
#    Uses _gauge_results saved by cell F.
#
#    Metrics computed per gauge:
#      NSE  = Nash-Sutcliffe Efficiency     (1=perfect, 0=mean model, <0=worse)
#      KGE  = Kling-Gupta Efficiency        (1=perfect; splits into r, beta, gamma)
#      rRMSE= RMSE / mean_obs               (scale-free, % error)
#      ratio= mean_sim / mean_obs           (magnitude bias)
#      r    = Pearson correlation           (timing quality)
#      lag  = month of max cross-correlation (0=in phase, +N=model lags N months)
#
#    Seasonal RMSE heatmap: reveals if errors concentrate in
#    spring snowmelt (MAM) or summer baseflow recession (JJA).
# ============================================================
from scipy.signal import correlate
import calendar

def _nse(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    if mask.sum() < 3:
        return np.nan
    o, s = obs[mask], sim[mask]
    return 1.0 - np.sum((s - o)**2) / np.sum((o - np.mean(o))**2)

def _kge(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    if mask.sum() < 3:
        return np.nan, np.nan, np.nan, np.nan
    o, s = obs[mask], sim[mask]
    r = float(np.corrcoef(o, s)[0, 1])
    beta  = float(np.mean(s) / np.mean(o))         # bias ratio
    gamma = float((np.std(s) / np.mean(s)) / (np.std(o) / np.mean(o)))  # variability ratio
    kge   = 1.0 - np.sqrt((r - 1)**2 + (beta - 1)**2 + (gamma - 1)**2)
    return float(kge), r, beta, gamma

def _lag_months(obs, sim, max_lag=6):
    mask = np.isfinite(obs) & np.isfinite(sim)
    if mask.sum() < 6:
        return 0
    o = (obs[mask] - obs[mask].mean())
    s = (sim[mask] - sim[mask].mean())
    xc = np.correlate(o, s, mode="full")
    mid = len(xc) // 2
    search = xc[mid - max_lag : mid + max_lag + 1]
    return int(np.argmax(search)) - max_lag   # negative=model leads, positive=model lags

# Build results table
_rows = []
_sim_date_idx = pd.DatetimeIndex(dates)
SEASONS = {"DJF": [12, 1, 2], "MAM": [3, 4, 5], "JJA": [6, 7, 8], "SON": [9, 10, 11]}

for _site, _res in _gauge_results.items():
    if _res["obs"] is None or len(_res["obs"]) == 0:
        continue
    _obs_df = _res["obs"].set_index("datetime")["BF_cms"]
    _sim_s  = pd.Series(_res["sim"], index=_sim_date_idx)

    # align on common months
    _common = _obs_df.index.intersection(_sim_s.index)
    if len(_common) < 12:
        continue
    _o = _obs_df.reindex(_common).values
    _s = _sim_s.reindex(_common).values

    _nse_v        = _nse(_o, _s)
    _kge_v, _r, _beta, _gamma = _kge(_o, _s)
    _mask = np.isfinite(_o) & np.isfinite(_s)
    _rmse_v  = float(np.sqrt(np.mean((_s[_mask] - _o[_mask])**2))) if _mask.sum() > 0 else np.nan
    _mean_o  = float(np.nanmean(_o))
    _rrmse_v = _rmse_v / _mean_o if _mean_o > 0 else np.nan
    _ratio_v = float(np.nanmean(_s)) / _mean_o if _mean_o > 0 else np.nan
    _lag_v   = _lag_months(_o, _s)

    # seasonal RMSE
    _sea_rmse = {}
    _dates_c  = pd.DatetimeIndex(_common)
    for _sname, _mos in SEASONS.items():
        _msk_s = np.array([d.month in _mos for d in _dates_c]) & _mask
        if _msk_s.sum() > 2:
            _sea_rmse[_sname] = float(np.sqrt(np.mean((_s[_msk_s] - _o[_msk_s])**2)))
        else:
            _sea_rmse[_sname] = np.nan

    _rows.append({
        "site": _site, "name": _res["name"], "n_drn": _res["n_drn"],
        "n_months": int(_mask.sum()),
        "mean_obs_cms": round(_mean_o, 2),
        "mean_sim_cms": round(float(np.nanmean(_s)), 2),
        "ratio": round(_ratio_v, 3),
        "r": round(_r, 3),
        "NSE": round(_nse_v, 3),
        "KGE": round(_kge_v, 3),
        "KGE_beta":  round(_beta, 3),
        "KGE_gamma": round(_gamma, 3),
        "RMSE_cms": round(_rmse_v, 2),
        "rRMSE_pct": round(_rrmse_v * 100, 1),
        "lag_months": _lag_v,
        **{f"RMSE_{k}": round(v, 2) for k, v in _sea_rmse.items()},
    })

_diag_df = pd.DataFrame(_rows).sort_values("NSE", ascending=False)
print(_diag_df[["name", "n_months", "mean_obs_cms", "mean_sim_cms", "ratio",
               "r", "NSE", "KGE", "rRMSE_pct", "lag_months"]].to_string(index=False))

_diag_csv = os.path.join(fig_dir, "stream_gauge_diagnostics.csv")
_diag_df.to_csv(_diag_csv, index=False)
print(f"\nDiagnostic table saved: {_diag_csv}")

# ── Figure A: NSE bar chart with KGE overlay ─────────────────────────────────
_df_plot = _diag_df.sort_values("NSE")
fig, axes = plt.subplots(1, 2, figsize=(15, 0.5 * len(_df_plot) + 2.5),
                         constrained_layout=True)

# Left: NSE bars coloured by quality band
_nse_vals = _df_plot["NSE"].values
_kge_vals = _df_plot["KGE"].values
_colors_nse = ["#4caf50" if v > 0.5 else "#ff9800" if v > 0.0 else "#f44336"
               for v in _nse_vals]
_ax = axes[0]
_ax.barh(_df_plot["name"], _nse_vals, color=_colors_nse, edgecolor="white", linewidth=0.5)
_ax.scatter(_kge_vals, _df_plot["name"], color="black", zorder=5,
            s=30, marker="D", label="KGE")
_ax.axvline(0.5, color="green",  linestyle="--", linewidth=1, alpha=0.6, label="NSE=0.5")
_ax.axvline(0.0, color="orange", linestyle="--", linewidth=1, alpha=0.6, label="NSE=0.0")
_ax.set_xlabel("Efficiency", fontsize=10)
_ax.set_title("NSE (bars)  +  KGE (diamonds)\nGreen > 0.5, Orange 0–0.5, Red < 0",
              fontsize=10, fontweight="bold")
_ax.legend(fontsize=8, loc="lower right")
_ax.grid(axis="x", alpha=0.3)

# Right: rRMSE and lag bubble
_ax2 = axes[1]
_scatter = _ax2.scatter(
    _df_plot["rRMSE_pct"], _df_plot["name"],
    c=_df_plot["lag_months"], cmap="RdBu_r", vmin=-4, vmax=4,
    s=80, edgecolors="black", linewidths=0.6, zorder=5
)
plt.colorbar(_scatter, ax=_ax2, label="Lag (months;  +ve = model lags obs)")
_ax2.axvline(30, color="orange", linestyle="--", linewidth=1, alpha=0.7, label="30% error")
_ax2.axvline(50, color="red",    linestyle="--", linewidth=1, alpha=0.7, label="50% error")
_ax2.set_xlabel("Relative RMSE (% of mean observed baseflow)", fontsize=10)
_ax2.set_title("Scale-free RMSE  +  Timing Lag\n(colour: +ve = model lags, -ve = model leads)",
               fontsize=10, fontweight="bold")
_ax2.legend(fontsize=8, loc="lower right")
_ax2.grid(axis="x", alpha=0.3)

fig.suptitle(f"Stream Gauge Diagnostics — {nameModel}", fontsize=13, fontweight="bold")
_out_diag_fig = os.path.join(fig_dir, "stream_gauge_diagnostics_metrics.png")
plt.savefig(_out_diag_fig, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", _out_diag_fig)

# ── Figure B: seasonal RMSE heatmap ──────────────────────────────────────────
_sea_cols = ["RMSE_DJF", "RMSE_MAM", "RMSE_JJA", "RMSE_SON"]
_sea_avail = [c for c in _sea_cols if c in _diag_df.columns]
if _sea_avail:
    _heat = _diag_df.set_index("name")[_sea_avail]
    # normalise each row by mean_obs so heatmap is relative (rRMSE per season)
    _mean_o_idx = _diag_df.set_index("name")["mean_obs_cms"]
    _heat_rel = _heat.divide(_mean_o_idx, axis=0) * 100   # %
    _heat_rel.columns = [c.replace("RMSE_", "") for c in _sea_avail]

    fig2, ax2 = plt.subplots(figsize=(7, max(5, 0.4 * len(_heat_rel) + 2)),
                             constrained_layout=True)
    _im = ax2.imshow(_heat_rel.values, aspect="auto", cmap="YlOrRd",
                     vmin=0, vmax=80)
    plt.colorbar(_im, ax=ax2, label="Seasonal rRMSE (%)", shrink=0.7)
    ax2.set_xticks(range(len(_heat_rel.columns)))
    ax2.set_xticklabels(_heat_rel.columns, fontsize=10)
    ax2.set_yticks(range(len(_heat_rel.index)))
    ax2.set_yticklabels(_heat_rel.index, fontsize=8)
    for _ii in range(len(_heat_rel.index)):
        for _jj in range(len(_heat_rel.columns)):
            _v = _heat_rel.values[_ii, _jj]
            if np.isfinite(_v):
                ax2.text(_jj, _ii, f"{_v:.0f}%", ha="center", va="center",
                         fontsize=7, color="black" if _v < 50 else "white")
    ax2.set_title(f"Seasonal Relative RMSE (% of mean observed baseflow)\n{nameModel}",
                  fontsize=11, fontweight="bold")
    _out_heat = os.path.join(fig_dir, "stream_gauge_seasonal_rmse.png")
    plt.savefig(_out_heat, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", _out_heat)

# ── Explain what to look for ──────────────────────────────────────────────────
print("\n=== HOW TO INTERPRET ===")
print("NSE > 0.5         : model skill better than using the long-term mean -- ACCEPTABLE")
print("NSE 0.0 to 0.5    : model has skill but large errors -- MARGINAL")
print("NSE < 0.0         : model is WORSE than just predicting the mean -- PROBLEMATIC")
print("lag_months +1/+2  : model responds to recharge 1-2 months LATE (normal for thick vadose zone)")
print("lag_months >+3    : model groundwater response too slow -- may need higher Kv or RCH_MULT")
print("ratio >> 1        : model over-predicts baseflow -- reduce RCH_MULT further")
print("ratio << 1        : model under-predicts -- consider if watershed is fully in model domain")
print("MAM RMSE highest  : snowmelt timing not captured well (NLDAS Qsb smooths out peaks)")
print("JJA RMSE highest  : summer recession too fast/slow -- drain conductance issue")


### H) Stream Network & Gauge Location Map
Overlays the NHD stream network on the mean depth-to-water map and marks each
USGS gauge with its NSE score so you can see spatial patterns in model performance.

In [ ]:
# ============================================================
# H) STREAM NETWORK + GAUGE LOCATIONS OVERLAY MAP
#
#    Shows:
#      - Mean depth-to-water-table (background raster)
#      - NHD stream network (blue lines)
#      - USGS gauge locations coloured by NSE score
#      - Model domain boundary
#
#    Why: reveals if poor-performing gauges are in areas
#    where the simulated water table is too deep or too shallow.
# ============================================================
import fiona, geopandas as gpd
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from pyproj import Transformer as _Tfm

# ── 1. Load NHD streams ───────────────────────────────────────────────────────
print("Loading NHD streams from GDB...")
_streams = gpd.read_file(gdb_path, layer=layer_name)
print(f"  {len(_streams):,} stream features loaded")

# ── 2. Compute mean depth-to-water over all periods ──────────────────────────
print("Computing mean depth-to-water...")
_active2d = idomain[0] > 0
_dtw_sum  = np.zeros((nrow, ncol), dtype=np.float64)
_dtw_cnt  = np.zeros((nrow, ncol), dtype=np.int32)
for _kk in range(head.shape[0]):
    _wt  = get_water_table(head[_kk], idomain, huge=1e29)
    _dtw = top - _wt
    _valid = _active2d & np.isfinite(_dtw) & (_dtw > -50) & (_dtw < 500)
    _dtw_sum += np.where(_valid, _dtw, 0.0)
    _dtw_cnt += _valid.astype(np.int32)
_mean_dtw = np.where(_dtw_cnt > 0, _dtw_sum / _dtw_cnt, np.nan)
print(f"  Mean DTW range: {np.nanmin(_mean_dtw):.1f} to {np.nanmax(_mean_dtw):.1f} m")

# ── 3. Gauge locations from _gauge_results + _diag_df ─────────────────────────
_gauge_pts = []
_tfm_wgs_to_mod = _Tfm.from_crs("EPSG:4326", f"EPSG:{EPSG}", always_xy=True)
for _site, _res in _gauge_results.items():
    # look up NSE from diagnostic table
    _row = _diag_df[_diag_df["site"] == _site]
    _nse_val = float(_row["NSE"].values[0]) if len(_row) > 0 else np.nan
    # gauge location from KEY_GAUGES (use bbox centre as approx)
    _g_cfg  = next((g for g in KEY_GAUGES if g["site"] == _site), None)
    if _g_cfg is None:
        continue
    _bx = _g_cfg["bbox"]
    _lon_c = (_bx[0] + _bx[2]) / 2
    _lat_c = (_bx[1] + _bx[3]) / 2
    _x_mod, _y_mod = _tfm_wgs_to_mod.transform(_lon_c, _lat_c)
    _gauge_pts.append({"site": _site, "name": _res["name"],
                        "NSE": _nse_val, "x": _x_mod, "y": _y_mod})

_gauge_pts_df = pd.DataFrame(_gauge_pts)

# Try to get exact gauge coordinates from USGS sites CSV if it exists
_sites_csv = os.path.join(fig_dir, "usgs_gauges_in_boundary.csv")
if os.path.exists(_sites_csv):
    _sites_csv_df = pd.read_csv(_sites_csv, dtype=str)
    _sites_csv_df["lon"] = pd.to_numeric(_sites_csv_df["dec_long_va"], errors="coerce")
    _sites_csv_df["lat"] = pd.to_numeric(_sites_csv_df["dec_lat_va"],  errors="coerce")
    _sites_csv_df = _sites_csv_df.dropna(subset=["lon", "lat"])
    for _idx, _pt in _gauge_pts_df.iterrows():
        _match = _sites_csv_df[_sites_csv_df["site_no"] == _pt["site"]]
        if len(_match) > 0:
            _lon = float(_match["lon"].values[0])
            _lat = float(_match["lat"].values[0])
            _x_mod, _y_mod = _tfm_wgs_to_mod.transform(_lon, _lat)
            _gauge_pts_df.loc[_idx, "x"] = _x_mod
            _gauge_pts_df.loc[_idx, "y"] = _y_mod
    print(f"Gauge coordinates updated from USGS site list")

# ── 4. Plot ───────────────────────────────────────────────────────────────────
_extent = get_extent(xorigin, yorigin, delr, delc, nrow, ncol)
_dtw_bounds = [-30, 0, 7, 25, 50, 100, 250, 500]
_dtw_colors = ["#c62828", "#d9f2ff", "#8be4ff", "#2fc7f0",
               "#10a8d2", "#087f9f", "#045a75"]
_dtw_cmap = ListedColormap(_dtw_colors)
_dtw_norm = BoundaryNorm(_dtw_bounds, _dtw_cmap.N)

gdf_bdry = gpd.read_file(boundary_shp)

fig, ax = plt.subplots(figsize=(13, 10), constrained_layout=True)

# Background: mean DTW
_dtw_show = np.where(idomain[0] > 0, _mean_dtw, np.nan)
_im = ax.imshow(_dtw_show, extent=_extent, origin="upper",
                cmap=_dtw_cmap, norm=_dtw_norm, alpha=0.85, zorder=1)
plt.colorbar(_im, ax=ax, label="Mean Depth-to-Water (m)", shrink=0.7,
             ticks=_dtw_bounds)

# Stream network
_streams.plot(ax=ax, color="#1565C0", linewidth=0.4, alpha=0.6, zorder=2,
              label="NHD streams")

# Domain boundary
gdf_bdry.boundary.plot(ax=ax, color="black", linewidth=0.9, zorder=3)

# Gauge scatter coloured by NSE
_nse_cmap = plt.cm.RdYlGn
_nse_norm = Normalize(vmin=-0.5, vmax=1.0)
_nse_sm   = ScalarMappable(cmap=_nse_cmap, norm=_nse_norm)
_nse_sm.set_array([])

_valid_pts = _gauge_pts_df.dropna(subset=["x", "y"])
_sc = ax.scatter(
    _valid_pts["x"], _valid_pts["y"],
    c=_valid_pts["NSE"].fillna(-1), cmap=_nse_cmap, norm=_nse_norm,
    s=90, edgecolors="black", linewidths=0.8, zorder=5,
    label="USGS gauge (colour = NSE)"
)
plt.colorbar(_sc, ax=ax, label="NSE (green > 0.5 = good)", shrink=0.5,
             location="left")

# Label gauges
for _, _pt in _valid_pts.iterrows():
    _short = _pt["name"].split(",")[0][:15]
    _nse_txt = f"NSE={_pt['NSE']:.2f}" if np.isfinite(_pt["NSE"]) else ""
    ax.annotate(
        f"{_short}\n{_nse_txt}",
        xy=(_pt["x"], _pt["y"]),
        xytext=(6, 4), textcoords="offset points",
        fontsize=6, color="black",
        bbox=dict(facecolor="white", alpha=0.6, linewidth=0, pad=1.5),
        zorder=6,
    )

ax.set_title(
    f"Mean Depth-to-Water + NHD Streams + USGS Gauge Performance\n"
    f"{nameModel}  |  Red = model fails, Green = model passes NSE",
    fontsize=12, fontweight="bold"
)
ax.set_xlabel("Easting (m, EPSG:3174)")
ax.set_ylabel("Northing (m, EPSG:3174)")
ax.legend(loc="lower right", fontsize=9)

_out_stream_map = os.path.join(fig_dir, "stream_network_dtw_gauge_nse_map.png")
plt.savefig(_out_stream_map, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", _out_stream_map)
